# Visual Asset Auditing System - Test Bench

This notebook implements the **Test Bench** tier of the Visual Asset Auditing System. It demonstrates:

1. Connecting to AlloyDB and GCS.
2. Translating text and/or an uploaded reference into visual, semantic, and tag retrieval signals.
3. Running exact content-hash lookup plus image-vector, text-vector, FTS, and Vision-tag search.
4. Deduplicating each arm by `content_hash` before RRF, then retaining corroborated candidates through drop-off and reranking.
5. Auditing a bounded set of unique image representatives with Gemini, then paging every physical instance of verified content hashes.
6. Creating and updating the `audit_results` table in AlloyDB.

*Transcribed from IMG_9422.jpeg. Additional source screenshots will be added below in the order received.*

In [ ]:
# Install required libraries
!pip install -q google-genai google-cloud-vision google-cloud-storage google-cloud-aiplatform google-cloud-alloydb-connector kfp google-cloud-pipeline-components pgvector asyncpg kneed pandas numpy pillow nest-asyncio sqlalchemy "protobuf<5.0.0dev"


## Package Installation

This cell installs all necessary external libraries (Google GenAI, Cloud Vision, AI Platform, pgvector, asyncio) to equip the notebook environment.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Google Cloud, Vertex AI & AlloyDB Configuration

This is the notebook's single configuration cell. It uses the existing global variables or same-named environment variables, authenticates through the Google Cloud session, and creates the one shared Vertex AI GenAI client. It does not use an API key.

In [ ]:
# Google Cloud, Vertex AI and AlloyDB configuration. Fill the project, model, database and bucket values below.
import os
os.environ["GOOGLE_API_USE_CLIENT_CERTIFICATE"] = "false"

PROJECT_ID = ""
REGION = ""

GEMINI_ORCHESTRATOR_MODEL = ""
GEMINI_INFERENCE_MODEL = ""
GEMINI_CROSS_ENCODER_MODEL = ""
EMBEDDING_MODEL = "gemini-embedding-2"

ALLOYDB_CLUSTER = ""
ALLOYDB_INSTANCE = ""
DB_USER = "postgres"
DB_NAME = ""
DB_PASSWORD = ""
DB_SCHEMA = "visual_asset"

GCS_BUCKET = ""

import google.genai as genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")


## Google Cloud Authentication

This cell handles authentication with Google Cloud using Colab auth utilities, sets up API project client.

In [ ]:
# AlloyDB Connection Setup using SQLAlchemy Pool + AsyncConnector
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}

async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Establishes and pools AlloyDB connections using SQLAlchemy and the AsyncConnector, with automatic timeout diagnostics."""
    global _engine_cache
    global _connector_cache

    if reuse and 'default' in _engine_cache:
        return _engine_cache['default'], _connector_cache['default']

    # Use lazy refresh for serverless/Colab environments
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        # Handle case where user pasted the full resource path or just the instance ID
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"

        try:
            # Enforce 10-second connection timeout to prevent hanging loop CancelledErrors
            conn = await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=False, # Set to True if using IAM auth
                    ip_type=IPTypes.PUBLIC # Adjust to PUBLIC or PRIVATE
                ),
                timeout=10.0
            )
            return conn
        except asyncio.TimeoutError:
            raise ConnectionError(
                f"AlloyDB connection timed out (10s) to {instance_uri}. "
                "Ensure your client IP is authorized in the AlloyDB Public IP console, "
                "or check your VPC network access if running internally."
            )
        except Exception as e:
            raise ConnectionError(f"Failed to connect to AlloyDB: {e}")

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=10,
        max_overflow=20,
        pool_pre_ping=True # Force SQLAlchemy to health-check connections
    )

    if reuse:
        _engine_cache['default'] = engine
        _connector_cache['default'] = connector

    return engine, connector


In [ ]:
# # Run the setup
# import nest_asyncio
# nest_asyncio.apply()
# try:
#     asyncio.run(setup_database())
# except Exception as e:
#     print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# Database Connection Verification (Read-Only Check – No DDL or Schema Changes)
import asyncio
import nest_asyncio

async def verify_database_connection():
    """Verifies the connection and prints the retrieval-column types/indexes needed by the notebook. Read-only; no DDL."""
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            count = await db.fetchval(f"SELECT COUNT(*) FROM {DB_SCHEMA}.visual_assets;")
            print(f"Connected to AlloyDB successfully. Schema '{DB_SCHEMA}.visual_assets' is ready ({count} assets found).")
            critical_columns = ['asset_id', 'content_hash', 'embedding', 'vision_tags', 'gemini_description', 'asset_filename', 'gcs_raw_path', 'page_url', 'format']
            column_rows = await db.fetch(
                '''SELECT column_name, data_type, udt_name
                   FROM information_schema.columns
                   WHERE table_schema = $1 AND table_name = 'visual_assets'
                     AND column_name = ANY($2::text[])
                   ORDER BY column_name''',
                DB_SCHEMA, critical_columns,
            )
            print('[Schema diagnostic] Retrieval column types:')
            for row in column_rows:
                print(f"  - {row['column_name']}: {row['data_type']} ({row['udt_name']})")
            index_rows = await db.fetch(
                '''SELECT indexname, indexdef
                   FROM pg_indexes
                   WHERE schemaname = $1 AND tablename = 'visual_assets'
                   ORDER BY indexname''',
                DB_SCHEMA,
            )
            print('[Schema diagnostic] visual_assets indexes:')
            for row in index_rows:
                print(f"  - {row['indexname']}: {row['indexdef']}")
    except Exception as e:
        print(f"Database connection status: {e}")

# Run connection check
nest_asyncio.apply()
try:
    asyncio.run(verify_database_connection())
except Exception as e:
    print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# 1. Audit Config Generator & Embedding Generation
import json
import asyncio
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Tuple
from pydantic import BaseModel, Field

# Define Pydantic model for the structured Audit Context.
class AuditContextModel(BaseModel):
    audit_goal: str = Field(description="Refined, precise version of the user's goal.")
    evaluation_strategy: str = Field(description="Exactly one of: content_hash_exact (identical uploaded bytes only), visual_reference_variant (same visual identity allowing the stated variants), or semantic_visual_presence (a subject/category is visibly present).")
    primary_visual_signatures: List[str] = Field(description="Two to six observable, discriminative visual signatures that must be checked before a True verdict. Use shape, relative geometry, wordmark spelling/layout, face/subject attributes, or component structure; avoid generic color-only statements.")
    allowed_variations: List[str] = Field(description="Only the concrete changes that remain a match for this goal, such as crop, scale, compression, background placement, gradient, or color. Empty when no variation is allowed.")
    disqualifying_confusions: List[str] = Field(description="Two to five specific visual near-misses that must produce False, such as a different person, a related logo, a different wordmark, or a generic icon.")
    image_description: Optional[str] = Field(None, description="Description of the reference image if provided, else null.")
    reference_is_composite_canvas: bool = Field(description="True if the reference image is a composite layout, webpage screenshot, hero banner, or real-world photograph containing the logo/asset. False if the reference image represents an isolated, standalone logo on a plain background.")
    inclusion_criteria: List[str] = Field(description="List of 3-7 specific, testable criteria an image MUST meet to be relevant.")
    exclusion_criteria: List[str] = Field(description="List of 2-5 criteria that EXCLUDE an image from relevance.")
    adjudication_logic: str = Field(description="A clear IF-THEN-ELSE statement defining PASS/FAIL conditions.")
    search_keywords: List[str] = Field(description="List of 5-10 single-word search terms (e.g. ['woman', 'female', 'portrait']) rather than multi-word phrases, to ensure broad keyword match capability in indexed assets.")
    vision_tag_filter: List[str] = Field(description="List of 2-8 concise semantic tag hints for the target and close synonyms. Stage 1 will resolve them locally to exact database Vision tags; do not emit a large vocabulary list.")
    audit_instructions: str = Field(description="Detailed instructions to be passed to the auditing LLM describing the criteria.")
    extraction_schema: Dict[str, str] = Field(description="Dynamic key-value pairs representing additional boolean/integer/string properties to extract from the image to verify the audit criteria.")

def get_image_mime_type(path: str) -> str:
    """Helper to dynamically resolve visual asset MIME type based on file path extension."""
    lower_path = path.lower()
    if lower_path.endswith(".png"):
        return "image/png"
    elif lower_path.endswith(".webp"):
        return "image/webp"
    elif lower_path.endswith(".gif"):
        return "image/gif"
    return "image/jpeg"

def generate_audit_config(user_goal: str, reference_image_description: Optional[str] = None, available_tags: Optional[List[str]] = None) -> dict:
    """Translates a high-level user goal into a structured audit context with strict visual grounding and anti-hallucination guardrails."""
    image_context = ""
    if reference_image_description:
        image_context = f"\nReference Image Description: {reference_image_description}\n"

    # Never serialize the complete database tag catalogue into this model prompt. When a small curated list is supplied, use it; otherwise emit semantic hints that Stage 1 resolves locally against the catalogue.
    active_tags = available_tags if available_tags is not None else globals().get('WEB_AUDIT_VISION_TAGS', [])
    if active_tags:
        tag_vocabulary_instruction = f"""CRITICAL RULE FOR vision_tag_filter:\nYou MUST select 2-8 tags only from this small curated list: {', '.join(map(str, active_tags[:250]))}."""
    else:
        tag_vocabulary_instruction = """CRITICAL RULE FOR vision_tag_filter:\nReturn 2-8 concise semantic tag hints that describe the target, likely synonyms, and discriminative visual category. These are hints only: they will be resolved to exact database tags locally after this response. Do not return a large list."""

    builder_prompt = f"""
You are a Lead Enterprise Visual & UI/UX Asset Auditor building a high-precision audit configuration for executive leadership.
Your task is to expand a user's audit goal into an airtight, zero-mistake structured audit context.

User Goal: {user_goal}
{image_context}

BRAND COMPLIANCE SIGNATURES INFERENCE:
Analyze the User Goal and the Reference Image Description (if provided) to extract:
1. The target brand, UI component, or visual subject under audit (e.g. Google Pay, YouTube, a specific Favicon, a cookies banner, or Google Workspace logos).
2. What constitutes the COMPLIANT (active, modern, approved) visual design, layout, typography, or shape.
3. What constitutes the NON-COMPLIANT (legacy, outdated, spoofed, or incorrect) visual design, layout, typography, or shape.
Ground all inclusion and exclusion criteria strictly in these inferred compliance signatures.

STEP 1: DYNAMIC BRAND & INTENT EXTRACTION
Identify the target brand/visual subject under audit based on the brand compliance signatures inference. Restrict all criteria, search keywords, and instructions strictly to this extracted subject.
Set `evaluation_strategy` exactly as follows: `content_hash_exact` only when the user asks for byte-identical copies of the uploaded asset; `visual_reference_variant` when the same visual identity may appear with allowed recoloring, gradients, crop, scale, compression, or an old/new design; otherwise `semantic_visual_presence` for a subject/category such as a person, object, or scene. Do not use `content_hash_exact` merely because a reference image exists.
List 2-6 `primary_visual_signatures` that a visual evaluator can actually observe. List `allowed_variations` only when they are permitted by the user's goal. List `disqualifying_confusions` for the most likely false positives.

STRICT VISUAL GROUNDING & ANTI-HALLUCINATION (CRITICAL):
- You MUST base all inclusion/exclusion criteria and visual descriptions strictly and exclusively on what is physically visible inside the provided reference image.
- Do NOT assume, extrapolate, or hallucinate the presence of brand names, wordmarks, UI elements, button texts, or logos that are cropped out or missing from the reference image.
- If a button, text, or logo is not visible in the reference image, do NOT include it as a mandatory requirement (inclusion criteria).

STEP 2: COMPLIANCE STATE ALIGNMENT & TEMPLATE ROLE ANALYSIS
Analyze the role of the reference image template (if provided) using your inferred compliance signatures.
- **State Conflict (Negation/Comparative Match)**: If the reference image represents the *Active/Compliant/New* standard, but the user wants to find *outdated/old* assets.
  * The template role is **Negative / Comparative Match**.
  * The exclusion criteria MUST exclude the reference image's compliant visual signatures.
  * The inclusion criteria MUST target older legacy styles of that same brand.
- **State Alignment (Positive Template Match)**: If the reference image represents the *Outdated/Legacy/Old* standard, and the user wants to find *outdated/old* assets.
  * The template role is **Positive Template Match**.
  * The inclusion criteria MUST require matching the reference image's legacy visual signatures.
  * The exclusion criteria MUST explicitly exclude the modern, compliant standard.
- **General Discovery Match**: If the user wants to find *all* assets of the brand regardless of state.
  * The inclusion criteria should pass the reference style AND other iterations of the brand.

STEP 3: REFERENCE COMPOSITE CANVAS DETECTION
Evaluate the description of the reference image:
- Set `reference_is_composite_canvas` to True if it describes a composite scene, real-world photograph, or webpage screenshot containing the logo/asset.
- Set `reference_is_composite_canvas` to False only if it represents an isolated, standalone logo on a plain background.

STEP 4: STRICT BRAND EXCLUSION & ANTI-SPOOFING GUARDRAIL (CRITICAL)
If the target visual subject is a specific sub-brand or product logo:
- You MUST explicitly include in the `exclusion_criteria` a rule to exclude the generic corporate master logo unless it is explicitly accompanied by the sub-brand.
- ANTI-SPOOFING: Add explicit rules to reject look-alike misspellings or related but incorrect sub-brands.

STEP 5: COLOR & CANVAS CONTEXT INDEPENDENCE (CRITICAL)
Unless the user's goal explicitly specifies a color constraint, you MUST explicitly write in the `audit_instructions` and `adjudication_logic` that color is not a match determinant.

{tag_vocabulary_instruction}

CRITICAL RULE FOR search_keywords (CRITICAL):
- The search_keywords MUST be a list of single-word search terms rather than multi-word phrases.

CRITICAL MANDATES FOR OPTICAL RESOLUTION & CLARITY AUDITS (CRITICAL):
1. **Strict Adjudication Logic**: Write a crystal-clear IF-THEN-ELSE statement in `adjudication_logic`.
   - Recognize Cropped/Blurred Targets: A crop, compression artifact, low resolution, or embedded placement is not by itself a failure when every required structural signature is still visible.
   - Flag Resolution Status: In such cases, mark `optical_resolution_sufficient = False` in the extraction schema while still applying every inclusion and exclusion criterion.
   - Set `matches_criteria = False` whenever a required inclusion/signature is absent, an exclusion or disqualifying confusion is visible, the subject is different, or the available pixels are insufficient to verify the required signatures.
2. **Diagnostic Extraction Schema**: In `extraction_schema`, define:
   - `detected_asset_style`: string (Exact description of what is seen on canvas)
   - `is_outdated_or_noncompliant`: boolean
   - `is_embedded_in_composite_hero`: boolean
   - `composite_location_notes`: string
   - `optical_resolution_sufficient`: boolean

MANDATE FOR `audit_instructions`: Write an evidence-first Boolean decision protocol. It must require the evaluator to identify the target, verify every primary visual signature, allow only the stated variations, reject every listed confusion/exclusion, and return `matches_criteria=true` only when the observable pixels support all required criteria. Retrieval rank, filename, tags, or a generic semantic resemblance must never by themselves cause a True verdict.
"""

    # Passing the Pydantic model directly to the SDK
    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=builder_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AuditContextModel,
            temperature=0.0
        )
    )
    return json.loads(response.text)

def build_fused_query_text(context_dict: dict, is_negative: bool = False) -> str:
    """Combines all context fields into a single rich text representation for embedding."""
    if is_negative:
        return f"Exclude images that: {'; '.join(context_dict.get('exclusion_criteria', []))}. Specifically exclude spoofed misspellings, lookalike brands, and other products."

    parts = [
        f"Audit goal: {context_dict.get('audit_goal')}",
        f"Evaluation strategy: {context_dict.get('evaluation_strategy', 'semantic_visual_presence')}",
        f"Primary visual signatures: {'; '.join(context_dict.get('primary_visual_signatures', []))}",
        f"Include images that: {'; '.join(context_dict.get('inclusion_criteria', []))}",
        f"Allowed variations: {'; '.join(context_dict.get('allowed_variations', []))}",
    ]
    if context_dict.get("image_description"):
        parts.append(f"Reference analysis: {str(context_dict.get('image_description'))[:350]}")
    parts.append("For similarity, treat color and gradient changes as secondary only when allowed. Prioritize shape, text layout, logo structure, faces, people, and semantic subject.")
    return " | ".join(parts)

async def embed_audit_context(context_dict: dict, reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
    """Generates a POSITIVE and NEGATIVE multimodal embedding for Contrastive Retrieval."""
    pos_text = build_fused_query_text(context_dict, is_negative=False)
    neg_text = build_fused_query_text(context_dict, is_negative=True)

    pos_contents = []
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            pos_contents.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=mime))
        else:
            with open(reference_image_path, "rb") as f:
                pos_contents.append(types.Part.from_bytes(data=f.read(), mime_type=mime))

    pos_contents.append(pos_text[:1000])
    neg_contents = [neg_text[:1000]]

    def _embed(contents):
        return client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=contents,
            config=types.EmbedContentConfig(output_dimensionality=768)
        ).embeddings[0].values

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=2) as executor:
        pos_future = loop.run_in_executor(executor, _embed, pos_contents)
        neg_future = loop.run_in_executor(executor, _embed, neg_contents)
        pos_vec, neg_vec = await asyncio.gather(pos_future, neg_future)

    return pos_vec, neg_vec


In [ ]:
# 2. Parallel Hybrid Search with RRF & Drop-Off Detection
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from kneed import KneeLocator
from pgvector.asyncpg import register_vector
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def run_semantic_reranking_and_filter(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_context: dict, max_workers: int = 25) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies LLM Cross-Encoder semantic scoring to the pre-segmented candidates, with visual vector safeguards to protect recall against description gaps."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")
    audit_goal = audit_context.get("audit_goal", "")

    def _score_candidate(row):
        # 1. Visual Vector Safeguard Check (Absolute visual distance check)
        # Cosine distance < 0.28 means high visual similarity (approx >0.72 similarity).
        # If this is triggered, we flag it to bypass Cross-Encoder checks.
        vec_dist = row.get("vector_distance", 1.0)
        if vec_dist < 0.28:
            return {
                **row,
                "cross_encoder_score": 100, # Max score to force-promote
                "relevance_score": row.get("relevance_score", 0.0) * 2.0, # Visual match boost
                "vector_safeguard_triggered": True
            }

        desc = row.get("gemini_description", "")
        tags = ", ".join(row.get("vision_tags") or [])
        filename = row.get("asset_filename", "")

        prompt = f"""You are a rapid relevance scoring engine.
Audit Goal: {audit_goal}
Candidate Description: {desc}
Candidate Tags: {tags}
Candidate Filename: {filename}

Score the candidate's textual relevance on a scale of 0 to 100 using this calibrated rubric:
- 80-100 (High): Direct matches to the target subject, clear presence of target visual elements, or standalone brand logos requested.
- 30-79 (Borderline): Contextual matches, related terms/brands, composite graphics containing the target, or candidate descriptions with some visual layout complexity.
- 0-29 (Low): Completely unrelated elements, different subjects (e.g. illustrations when looking for photos, different products, or unrelated graphics).

Output a single integer from 0 to 100 representing the score. Output NOTHING ELSE.
"""
        try:
            resp = client.models.generate_content(
                model=GEMINI_CROSS_ENCODER_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0)
            )
            score = int(resp.text.strip())
        except:
            score = 50 # Default neutral fallback

        ce_mult = max(0.1, score / 50.0)
        new_score = row.get("relevance_score", 0.0) * ce_mult

        return {
            **row,
            "cross_encoder_score": score,
            "relevance_score": new_score,
            "vector_safeguard_triggered": False
        }

    print(f"[Cross-Encoder] Semantic reranking of {len(candidates)} candidate assets...")
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tasks = [loop.run_in_executor(executor, _score_candidate, item) for item in candidates]
        reranked = await asyncio.gather(*tasks)

    high_list = []
    edge_list = []
    low_list = []

    for item in reranked:
        if item.get("vector_safeguard_triggered", False):
            filename = item.get("asset_filename") or item["gcs_raw_path"].split("/")[-1]
            dist = item.get("vector_distance", 0)
            print(f"🛡️ Vector Safeguard triggered: Force-promoted {filename[:40]} due to high visual similarity (distance: {dist:.4f})")
            high_list.append(item)
            continue

        score = item["cross_encoder_score"]
        if score >= 75:
            high_list.append(item)
        elif score >= 30:
            edge_list.append(item)
        else:
            low_list.append(item)

    df_high_final = pd.DataFrame(high_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if high_list else pd.DataFrame()
    df_edge_final = pd.DataFrame(edge_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if edge_list else pd.DataFrame()
    df_low_final = pd.DataFrame(low_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if low_list else pd.DataFrame()

    return df_high_final, df_edge_final, df_low_final

def compute_weighted_rrf_rerank(candidates: list, audit_context: dict) -> list:
    """Zero-latency in-memory multi-factor reranker. Executes in local CPU RAM (< 0.5ms) without external API overhead."""
    tag_filter = [t.lower().strip() for t in (audit_context.get("retrieval_tag_scope") or audit_context.get("vision_tag_filter", [])) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]
    reranked = []

    for item in candidates:
        row = item["data"]
        base_score = item["score"]
        multiplier = 1.0

        # 1. Vision tag exact hit boost (2.0x per matching tag up to 8x)
        tags = [str(t).lower() for t in (row.get("vision_tags") or [])]
        tag_hits = sum(1 for t in tags if any(ft in t or t in ft for ft in tag_filter))
        if tag_hits > 0:
            multiplier *= (2.0 ** min(tag_hits, 3))

        # 2. Keyword exact hit in filename or description boost (1.5x per matching keyword up to 2.25x)
        desc = str(row.get("gemini_description") or "").lower()
        fname = str(row.get("asset_filename") or "").lower()
        kw_hits = sum(1 for kw in search_keywords if kw in desc or kw in fname)
        if kw_hits > 0:
            multiplier *= min(1.5 ** kw_hits, 2.25)

        reranked.append({
            "data": row,
            "score": base_score * multiplier,
            "base_rrf_score": base_score,
            "rerank_multiplier": multiplier,
            "vector_distance": item.get("vector_distance", 1.0)
        })

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return reranked

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: int = 10000) -> List[dict]:
    """Performs parallel 3-Arm Vector + Keyword + Tag Boosting search with RRF fusion, local reranking, and deduplication (No silent cliff)."""
    pos_vec, neg_vec = await embed_audit_context(audit_context, reference_image_path)

    engine, _ = await get_alloydb_connection()

    search_keywords = audit_context.get("search_keywords", [])
    keyword_query_str = " OR ".join(search_keywords) if search_keywords else ""

    # Stage 1 resolves semantic hints to canonical database tags. Never use a raw or full tag vocabulary as a search filter.
    tag_filter = audit_context.get("retrieval_tag_scope") or audit_context.get("vision_tag_filter", [])
    tag_filter = tag_filter if tag_filter else []

    # Run the 3 database query arms concurrently using pooled SQLAlchemy connections.
    async def run_vector():
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            if tag_filter:
                rows = await db.fetch(
                    f"""SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                               (embedding <=> $1::vector) as vector_distance
                        FROM {DB_SCHEMA}.visual_assets
                        WHERE embedding IS NOT NULL AND vision_tags::text[] && $2::text[]
                        ORDER BY embedding <=> $1::vector
                        LIMIT $3""",
                    pos_vec, tag_filter, limit
                )
            else:
                rows = await db.fetch(
                    f"""SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                               (embedding <=> $1::vector) as vector_distance
                        FROM {DB_SCHEMA}.visual_assets
                        WHERE embedding IS NOT NULL
                        ORDER BY embedding <=> $1::vector
                        LIMIT $2""",
                    pos_vec, limit
                )
            return [dict(r) for r in rows]

    async def run_fts():
        if not keyword_query_str:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            if tag_filter:
                rows = await db.fetch(
                    f"""SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                        FROM {DB_SCHEMA}.visual_assets
                        WHERE vision_tags::text[] && $2::text[]
                          AND to_tsvector('english', coalesce(gemini_description::text, '')) @@ websearch_to_tsquery('english', $1)
                        LIMIT $3""",
                    keyword_query_str, tag_filter, limit
                )
            else:
                rows = await db.fetch(
                    f"""SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                        FROM {DB_SCHEMA}.visual_assets
                        WHERE to_tsvector('english', coalesce(gemini_description::text, '')) @@ websearch_to_tsquery('english', $1)
                        LIMIT $2""",
                    keyword_query_str, limit
                )
            return [dict(r) for r in rows]

    async def run_tags():
        if not tag_filter:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE vision_tags::text[] && $1::text[]
                LIMIT $2
                """,
                tag_filter, limit
            )
            return [dict(r) for r in rows]

    vector_task = run_vector()
    fts_task = run_fts()
    tag_task = run_tags()

    vector_results, fts_results, tag_results = await asyncio.gather(vector_task, fts_task, tag_task)

    promoted_ids = set()
    for r in fts_results[:100]:
        promoted_ids.add(str(r["asset_id"]))
    for r in tag_results[:100]:
        promoted_ids.add(str(r["asset_id"]))

    # 3-Arm RRF Fusion (k=60)
    k = 60
    results_map = {}

    # Pre-populate with vector distances
    vector_distance_map = {str(r["asset_id"]): r["vector_distance"] for r in vector_results}

    def upsert_ranks(results_list, weight=1.0):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                results_map[img_id] = {"data": row, "score": 0.0, "vector_distance": vector_distance_map.get(img_id, 1.0)}
            results_map[img_id]["score"] += weight / (k + rank + 1)

    upsert_ranks(vector_results, weight=0.60)
    if fts_results:
        upsert_ranks(fts_results, weight=0.25)
    if tag_results:
        upsert_ranks(tag_results, weight=0.15)

    fused = list(results_map.values())

    # Zero-Latency In-Memory Reranking (Provides a smooth score curve for Kneedle)
    reranked_fused = compute_weighted_rrf_rerank(fused, audit_context)

    # Deduplication
    seen_identifiers = set()
    deduplicated = []
    for item in reranked_fused:
        row = item["data"]
        img_id = str(row["asset_id"])
        img_identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if img_identifier not in seen_identifiers:
            seen_identifiers.add(img_identifier)
            is_promoted = img_id in promoted_ids
            deduplicated.append({
                **row,
                "relevance_score": item["score"],
                "base_rrf_score": item.get("base_rrf_score", 0),
                "rerank_multiplier": item.get("rerank_multiplier", 1),
                "promoted_by_keyword_or_tag": is_promoted,
                "vector_distance": item.get("vector_distance", 1.0)
            })

    return deduplicated[:limit]

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies Kneedle curvature + rolling volatility to segment candidates into High, Edge, and Low."""
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_sorted = df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)
    y = df_sorted["relevance_score"].values
    x = np.arange(len(y))

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        return df_sorted.iloc[:int(len(y)*0.3)], df_sorted.iloc[int(len(y)*0.3):int(len(y)*0.6)], df_sorted.iloc[int(len(y)*0.6):]

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = x / (len(x) - 1)

    coords = np.column_stack((x_norm, y_norm))
    line_start, line_end = coords[0], coords[-1]
    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
    vec_from_start = coords - line_start
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    proj_on_line = np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.sqrt(np.sum((coords - proj_on_line)**2, axis=1))

    idx1 = np.argmax(dist_to_line)

    window = max(3, int(len(y) * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)

    idx2 = len(y) - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_borderline_width = max(10, int((len(y) - idx1) * 0.25))
    if (idx2 - idx1) < min_borderline_width:
        idx2 = min(len(y) - 1, idx1 + min_borderline_width)

    idx1 = max(10, idx1)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1: idx2 + 1].copy()
    low_df = df_sorted.iloc[idx2 + 1:].copy()

    # Visual Vector Safeguard: Check if any candidate has extremely high similarity (distance < 0.28)
    # even if it is currently classified in Low_df (or has been discarded).
    # Force rescue these to protect visual recall.
    if "vector_distance" in low_df.columns:
        rescued_vec = low_df[low_df["vector_distance"] < 0.28].copy()
        if not rescued_vec.empty:
            edge_df = pd.concat([edge_df, rescued_vec], ignore_index=True)
            low_df = low_df[low_df["vector_distance"] >= 0.28].copy()
            print(f"🛡️ Vector Safeguard triggered in Kneedle: Force-rescued {len(rescued_vec)} candidate(s) from Low to Borderline based on high visual similarity.")

    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = low_df[low_df["promoted_by_keyword_or_tag"] == True].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df = low_df[low_df["promoted_by_keyword_or_tag"] != True].copy()
            print(f"🛡️ Safeguard triggered: Promoted {len(rescued)} composite/diluted candidates from Low to Borderline tier based on exact keyword/tag match.")

    return high_df, edge_df, low_df


## Retrieval, Reranking, and Drop-off Segmentation Engine

This cell defines the core search engine:

- `run_hybrid_search`: Combines exact-hash, image/text-vector, Full-Text, and Vision-tag arms into a per-content-hash fused RRF list.
- `detect_dropoff_flawless`: Uses two robust score boundaries for High, Borderline, and Low bands, rescuing only exact or corroborated tail candidates.
- `run_semantic_reranking_and_filter`: Applies fast local contextual reranking before image-level Gemini inference.

In [ ]:
# 3. Hydration & Parallel LLM Audit Inference
import asyncio
from concurrent.futures import ThreadPoolExecutor
import json
import pandas as pd
from typing import List, Tuple, Optional
from google.genai import types
from google.cloud import storage
from pydantic import create_model, Field
import time
from PIL import Image
import io

def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Detects alpha transparency in an image and composites it onto a solid dark background to ensure light elements/text remain visible."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
            img = img.convert('RGBA')
            # Create a solid dark grey background
            bg = Image.new("RGBA", img.size, default_bg + (255,))
            alpha_composite = Image.alpha_composite(bg, img)
            final_img = alpha_composite.convert("RGB")
            out_bytes = io.BytesIO()
            final_img.save(out_bytes, format="PNG")
            return out_bytes.getvalue()
    except Exception as e:
        print(f"Warning: Failed to preprocess image transparency: {e}")
    return image_bytes

def enforce_zero_false_positives_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Enforces zero-false-positive criteria. Demotes matches if confidence is below 75%."""
    if df.empty:
        return df

    def guardrail_check(row):
        matches = bool(row.get("matches_criteria", False))
        conf = int(row.get("match_confidence", 100))
        rationale = str(row.get("visual_analysis_step_by_step", "")) + " " + str(row.get("match_rationale", ""))
        if matches and conf < 75:
            row["matches_criteria"] = False
            row["match_rationale"] = f"[GUARDRAIL DEMOTION: Conf {conf}% < 75%] {rationale}"
        return row

    return df.apply(guardrail_check, axis=1)

async def run_llm_audit_single(asset_data: dict, audit_config: dict, _executor=None, reference_image_part: Optional[types.Part] = None) -> dict:
    """Evaluates a single image asset against dynamic JSON schema using Pydantic, supporting side-by-side reference comparisons and transparency blending."""
    audit_instructions = audit_config.get("audit_instructions", "")
    extraction_schema = audit_config.get("extraction_schema", {})
    inclusion_criteria = audit_config.get("inclusion_criteria", [])
    exclusion_criteria = audit_config.get("exclusion_criteria", [])
    adjudication_logic = audit_config.get("adjudication_logic", "")
    is_composite = bool(audit_config.get("reference_is_composite_canvas", False))

    inclusion_str = "\n".join([f"- {c}" for c in inclusion_criteria]) if inclusion_criteria else "- None specified"
    exclusion_str = "\n".join([f"- {c}" for c in exclusion_criteria]) if exclusion_criteria else "- None specified"

    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description="Concise observable visual evidence: describe the exact shapes, typography, brand marks, colors, and layout that support the verdict.")
        ),
        "matches_criteria": (
            bool,
            Field(description="Strict final evaluation: True ONLY if the asset matches the target criteria and passes adjudication_logic (even inside a composite hero banner). False otherwise.")
        ),
        "match_confidence": (
            int,
            Field(ge=0, le=100, description="Match Confidence percentage (0-100%). You must assign < 75 if there is any doubt or visual occlusion.")
        ),
        "match_rationale": (
            str,
            Field(description="A concise final executive rationale explaining exactly why matches_criteria evaluated to True or False based on the visual_analysis_step_by_step.")
        )
    }

    for field_name, field_info in extraction_schema.items():
        if field_name in ["matches_criteria", "match_confidence", "match_rationale", "visual_analysis_step_by_step"]:
            continue

        t = str
        desc = f"Extracted value for {field_name}"

        if isinstance(field_info, dict):
            ftype = field_info.get("field_type", "string").lower()
            desc = field_info.get("description", desc)
        else:
            ftype = str(field_info).lower()

        if ftype == "boolean":
            t = bool
        elif ftype == "integer":
            t = int
        elif ftype == "number":
            t = float

        fields[field_name] = (t, Field(description=desc))

    DynamicAuditModel = create_model("DynamicAuditModel", **fields)

    reference_instructions = ""
    if reference_image_part:
        if is_composite:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Target Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE (CRITICAL):
The reference image (image_0) is a **Composite Canvas** (a complex real-world photograph/screenshot containing the logo).
Do NOT expect the candidate image under audit (image_1) to contain the hands, terminals, backgrounds, or full layout seen in image_0.
Instead, look at the target logo/brand style (e.g. logos or wordmark shown on the phone screen) inside image_0.
Verify if the candidate image (image_1) contains that target logo style. Ignore all other background visual noise in image_0.
"""
        else:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Image Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE:
Since the Reference Image (image_0) is a standalone logo, compare the candidate image (image_1) side-by-side against image_0.
"""

    # Updated prompt template: Objective, clean, and safe from prompt override classifications
    audit_prompt = f"""
Evaluate this image against the specified audit goal and criteria with high precision.

{reference_instructions}

[Color Independence Rule]:
Unless the Inclusion Criteria explicitly mention a required color, you must ignore any color differences between the reference image and the candidate image.

[Exact Typography Rule]:
Pay strict attention to typography and spelling (e.g. 'Google Play' is NOT 'Google Pay', 'Ads' is NOT 'AdWords'). Reject any assets that contain lookalike or misspelled branding unless the inclusion criteria explicitly permit them.

AUDIT SCOPE & EVALUATION RULES:

[Inclusion Criteria – Asset MUST fulfill these to pass]:
{inclusion_str}

[Exclusion Criteria – If asset triggers any of these, it MUST fail]:
{exclusion_str}

[Strict Adjudication Rule]:
{adjudication_logic}

[General Audit Instructions]:
{audit_instructions}

EXECUTION STEPS:
1. Provide concise, observable visual evidence in the `visual_analysis_step_by_step` field. Scan the entire canvas for the shapes, text, and layout relevant to the criteria.
2. Extract all diagnostic visual features requested in the schema, including whether the target is embedded inside a composite hero graphic (`is_embedded_in_composite_hero`).
3. Apply the Strict Adjudication Rule against your extracted visual findings.
4. Set `matches_criteria` to true when the available visual evidence supports every required inclusion criterion and no exclusion is visible. Do not require perfect pixels: a cropped, compressed, or embedded target may still be true when its required structural signatures are visible. Use `match_confidence` to express image quality or residual uncertainty; return false only for a contradiction, a near miss, or genuinely insufficient evidence.
5. Provide a clear justification in `match_rationale` explaining your decision based on your step-by-step analysis.
"""

    gcs_path = asset_data["gcs_raw_path"]

    # Resolve bytes through the calibrated loader so every concurrent audit obeys the source-byte budget.
    def _download_and_preprocess():
        return process_transparency(_read_image_bytes(gcs_path))

    loop = asyncio.get_running_loop()
    try:
        # Download and composite image on background thread
        processed_bytes = await loop.run_in_executor(_executor, _download_and_preprocess)
        image_part = types.Part.from_bytes(data=processed_bytes, mime_type="image/png")

        contents = []
        if reference_image_part:
            contents.append(reference_image_part)

        contents.append(image_part)
        contents.append(audit_prompt)

        def _call_gemini():
            return client.models.generate_content(
                model=GEMINI_INFERENCE_MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=DynamicAuditModel,
                    temperature=0.0
                )
            )

        response = await loop.run_in_executor(_executor, _call_gemini)
        extracted_data = json.loads(response.text)
        raw_verdict = extracted_data.get('matches_criteria')
        if isinstance(raw_verdict, bool):
            extracted_data['matches_criteria'] = raw_verdict
        elif str(raw_verdict).casefold() in {'true', '1'}:
            extracted_data['matches_criteria'] = True
        elif str(raw_verdict).casefold() in {'false', '0'}:
            extracted_data['matches_criteria'] = False
        else:
            raise ValueError('Model response did not contain a Boolean matches_criteria verdict')
        extracted_data['match_confidence'] = max(0, min(100, int(extracted_data.get('match_confidence', 0))))
        extracted_data['evaluation_status'] = 'completed'
    except Exception as e:
        extracted_data = {
            "matches_criteria": False,
            "match_confidence": 0,
            "match_rationale": f"Audit evaluation failed due to error: {str(e)}",
            "evaluation_status": "error",
            "error": str(e)
        }

    return {**asset_data, **extracted_data}

async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_config: dict, max_workers: int = 15, reference_image_path: Optional[str] = None) -> pd.DataFrame:
    """Runs parallel multi-threaded LLM inference on candidate subsets, supporting side-by-side template matches, GCS/local transparency preprocessing, and caching."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    print(f"Starting parallel LLM audit inference on {len(candidates)} candidates (Max concurrency: {max_workers})...")

    # Helper to download and preprocess the reference image once
    def _get_reference_part():
        if reference_image_path.startswith("gs://"):
            bucket_name = reference_image_path.split("/")[2]
            blob_name = "/".join(reference_image_path.split("/")[3:])
            client_storage = storage.Client(project=PROJECT_ID)
            bucket = client_storage.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            ref_bytes = blob.download_as_bytes()
        else:
            with open(reference_image_path, "rb") as f:
                ref_bytes = f.read()
        processed_ref = process_transparency(ref_bytes)
        return types.Part.from_bytes(data=processed_ref, mime_type="image/png")

    t0 = time.time()
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        reference_image_part = None
        if reference_image_path:
            reference_image_part = await loop.run_in_executor(executor, _get_reference_part)

        tasks = [run_llm_audit_single(asset, audit_config, _executor=executor, reference_image_part=reference_image_part) for asset in candidates]
        results = await asyncio.gather(*tasks)

    duration = time.time() - t0
    print(f" Processing assets... [{len(candidates)}/{len(candidates)}] completed")
    print(f" Inference completed. Enforcing Zero-FP policies and sorting scoreboard...")

    results_df = pd.DataFrame(results)
    results_df = enforce_zero_false_positives_rules(results_df)

    results_df = results_df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)

    print(f"\n=== VERIFIED AUDIT SCOREBOARD (Sorted by Search Similarity) ===")
    for i, r in results_df.iterrows():
        status_label = "PASS" if r.get("matches_criteria", False) else "FAIL"
        filename = r.get("asset_filename") or r.get("gcs_raw_path", "").split("/")[-1]
        sim = r.get("relevance_score", 0.0)
        conf = r.get("match_confidence", 100)
        print(f"[{i+1}/{len(results_df)}] {status_label} | {filename[:40]} | Sim: {sim:.4f} | Conf: {conf}%")

    return results_df


In [ ]:
# 4. Calibration Summary & Audit Results Saving E2E Loop
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame):
    """Saves the final audited results into AlloyDB (Bypassed by default in the interactive runner)."""
    if results_df.empty:
        return

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        records = results_df.to_dict("records")
        for r in records:
            verdict = "PASS" if r.get("matches_criteria") is True else "FAIL"

            # Extract criteria_checks from JSON response or construct it
            criteria_checks = {k: v for k, v in r.items() if k not in ["asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"]}

            await db.execute(
                f"""
                INSERT INTO {DB_SCHEMA}.audit_results (
                    session_id, asset_id, overall_verdict, adjudication_result,
                    criteria_checks, rationale, confidence_band
                ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                """,
                session_id,
                r.get("asset_id"),
                verdict,
                r.get("matches_criteria", False),
                json.dumps(criteria_checks),
                r.get("match_rationale", "Completed"),
                "high" if r.get("match_confidence", 0) > 95 else ("borderline" if r.get("match_confidence", 0) >= 70 else "below_threshold")
            )
    print("Audit results saved to database.")

async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: dict) -> str:
    """Generates an AI-powered executive summary of the visual asset audit results."""
    import json
    import numpy as np
    if results_df is None or results_df.empty:
        return "No audit results available to generate a summary."

    # Select columns to pass to the LLM, avoiding internal or verbose vector columns
    exclude_cols = {"num_chunks", "max_relevance_score", "gcs_raw_path", "gcs_processed_path", "embedding", "embedding_at"}
    cols_to_include = [col for col in results_df.columns if col not in exclude_cols]

    # Convert to records safely, handling NumPy arrays, lists, and floats without boolean truth value ambiguity
    clean_df = results_df[cols_to_include].copy()
    for col in clean_df.columns:
        def safe_clean(val):
            if val is None:
                return None
            if isinstance(val, (np.ndarray, pd.Series)):
                return val.tolist() if val.size > 0 else None
            if isinstance(val, float) and pd.isna(val):
                return None
            return val
        clean_df[col] = clean_df[col].apply(safe_clean)

    records = clean_df.to_dict(orient="records")
    formatted_results = json.dumps(records, indent=2)

    audit_instructions = audit_config.get("audit_instructions", "No specific audit context provided.")

    # Compute basic stats to seed in the prompt
    total_audited = len(results_df)
    matches_col = "matches_criteria" if "matches_criteria" in results_df.columns else None
    if matches_col:
        # Convert to boolean safely, handling string representation if any
        matches_true = results_df[matches_col].apply(lambda x: str(x).lower() in ("true", "1", "yes")).sum()
    else:
        matches_true = "N/A"

    summary_prompt = f"""
You are a Lead Visual Asset Auditor.
Your task is to write a visually engaging, highly structured, and extremely concise summary of a visual asset audit Test Bench calibration run.

STRICT RULES FOR FORMATTING & SECTIONS:
1. You MUST only include the following exact three sections in the output:
   - Objective & Scope (Preview Subset) (within the top [!NOTE] block)
   - Calibration Statistics (as a numbered list)
   - Configuration Calibration Insights (as a single, brief narrative paragraph of 3-4 sentences detailing the visual rules performance)
2. DO NOT include any other sections.
3. DO NOT pass any definitive verdicts of success.

AUDIT CONTEXT (Visual Evaluation Criteria):
{audit_instructions}

TEST BENCH STATS:
- Total images audited: {total_audited}
- Total images matching criteria (True): {matches_true}

TEST BENCH FINDINGS (JSON):
{formatted_results}
"""

    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=summary_prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


## Telemetry Summaries & Calibration Database Saving

This cell defines functions to generate final executive AI summaries (`generate_ai_audit_summary`) and save audit session details into the run calibration tables in AlloyDB for telemetry history.

In [ ]:
# 5. E2E Execution Helpers (Interactive Split Stages)
import base64
import os
import time
import json
import re
from collections import defaultdict
import pandas as pd
from google.cloud import storage

# Stage 1 tag-scope resolver. The full catalogue stays local/DB-side and is never inserted into an LLM prompt.
TAG_SCOPE_TUNING = {
    'max_tag_hints': 16,
    'max_canonical_tags': 64,
    'generic_tokens': {'asset', 'design', 'face', 'graphic', 'human', 'image', 'people', 'person', 'photo', 'photograph', 'picture', 'visual'},
}
_TAG_VOCABULARY_CACHE = None
_TAG_SYNONYMS = {
    'male': {'male', 'man', 'men', 'boy', 'gentleman', 'masculine'},
    'man': {'male', 'man', 'men', 'boy', 'gentleman', 'masculine'},
    'female': {'female', 'woman', 'women', 'girl', 'lady', 'feminine'},
    'woman': {'female', 'woman', 'women', 'girl', 'lady', 'feminine'},
    'favicon': {'favicon', 'icon', 'glyph', 'symbol', 'logo', 'mark'},
    'icon': {'favicon', 'icon', 'glyph', 'symbol', 'logo', 'mark'},
    'portrait': {'portrait', 'face', 'person', 'headshot'},
    'logo': {'logo', 'logotype', 'wordmark', 'symbol', 'mark', 'icon'},
}
_TAG_CONFLICTS = {
    'male': {'female', 'woman', 'women', 'girl', 'lady', 'feminine'},
    'man': {'female', 'woman', 'women', 'girl', 'lady', 'feminine'},
    'female': {'male', 'man', 'men', 'boy', 'gentleman', 'masculine'},
    'woman': {'male', 'man', 'men', 'boy', 'gentleman', 'masculine'},
}

# Read lazily during Stage 1/2; this is not a runtime configuration preflight.
# Native arrays are queried without casting the column, so a matching GIN index remains usable.
_VISUAL_ASSET_COLUMN_TYPES_CACHE = None
_NATIVE_TAG_ARRAY_TYPES = {'text[]', 'character varying[]', 'varchar[]', 'citext[]'}

async def _get_visual_asset_column_types() -> dict:
    global _VISUAL_ASSET_COLUMN_TYPES_CACHE
    if _VISUAL_ASSET_COLUMN_TYPES_CACHE is not None:
        return _VISUAL_ASSET_COLUMN_TYPES_CACHE
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        rows = await db.fetch(
            '''SELECT attribute.attname AS column_name,
                      pg_catalog.format_type(attribute.atttypid, attribute.atttypmod) AS data_type
               FROM pg_catalog.pg_attribute AS attribute
               JOIN pg_catalog.pg_class AS relation ON relation.oid = attribute.attrelid
               JOIN pg_catalog.pg_namespace AS namespace ON namespace.oid = relation.relnamespace
               WHERE namespace.nspname = $1 AND relation.relname = 'visual_assets'
                 AND attribute.attname = ANY(CAST($2 AS text[]))
                 AND attribute.attnum > 0 AND NOT attribute.attisdropped''',
            DB_SCHEMA, ['vision_tags', 'content_hash', 'embedding', 'gemini_description', 'asset_filename'],
        )
    column_types = {str(row['column_name']): str(row['data_type']).casefold() for row in rows}
    _VISUAL_ASSET_COLUMN_TYPES_CACHE = column_types
    return column_types

def _native_tag_array_type(column_types: dict) -> Optional[str]:
    tag_type = str((column_types or {}).get('vision_tags') or '').casefold()
    return tag_type if tag_type in _NATIVE_TAG_ARRAY_TYPES else None

def _tag_filter_predicate(native_tag_type: Optional[str], parameter_number: int) -> Optional[str]:
    if not native_tag_type:
        return None
    return f'vision_tags && CAST(${parameter_number} AS {native_tag_type})'

def _tag_value_source(native_tag_type: Optional[str]) -> Optional[str]:
    if not native_tag_type:
        return None
    return 'CROSS JOIN LATERAL unnest(vision_tags) AS tag_value(tag)'


def _normalise_tag_text(value: str) -> str:
    return ' '.join(re.findall(r'[a-z0-9]+', str(value or '').casefold()))

def _tag_hint_terms(audit_config: dict) -> list:
    ordered = []
    for raw in list(audit_config.get('vision_tag_filter', [])) + list(audit_config.get('search_keywords', [])):
        normalised = _normalise_tag_text(raw)
        if normalised and normalised not in ordered:
            ordered.append(normalised)
    return ordered[:TAG_SCOPE_TUNING['max_tag_hints']]

def _expanded_tag_hint_terms(hints: list) -> list:
    terms = []
    for hint in hints:
        for token in hint.split():
            for expanded in _TAG_SYNONYMS.get(token, {token}):
                if expanded not in terms:
                    terms.append(expanded)
    return terms

def _explicit_exact_bytes_mode(user_goal: str) -> bool:
    '''Only an explicit byte/file/hash duplicate request may bypass image-level evaluation.'''
    goal = _normalise_tag_text(user_goal)
    exact_phrases = ('byte identical', 'identical bytes', 'same bytes', 'content hash', 'same file', 'exact file', 'duplicate file', 'exact duplicate')
    return any(phrase in goal for phrase in exact_phrases)

def _build_tag_vocabulary(rows) -> dict:
    entries, exact, token_index = [], {}, defaultdict(list)
    for row in rows:
        tag = str(row['tag']).strip()
        normalised = _normalise_tag_text(tag)
        if not normalised or normalised in exact:
            continue
        index = len(entries)
        tokens = frozenset(normalised.split())
        entries.append({'tag': tag, 'normalised': normalised, 'tokens': tokens})
        exact[normalised] = index
        for token in tokens:
            token_index[token].append(index)
    return {'entries': entries, 'exact': exact, 'token_index': token_index}

async def _load_tag_vocabulary() -> dict:
    global _TAG_VOCABULARY_CACHE
    if _TAG_VOCABULARY_CACHE is not None:
        return _TAG_VOCABULARY_CACHE
    column_types = await _get_visual_asset_column_types()
    native_tag_type = _native_tag_array_type(column_types)
    value_source = _tag_value_source(native_tag_type)
    if not value_source:
        print(f"[Tag scope] vision_tags is {column_types.get('vision_tags', 'unknown')!r}; native-array tag scope is disabled and Stage 2 will fail open to global retrieval.")
        return {'entries': [], 'exact': {}, 'token_index': defaultdict(list)}
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        rows = await db.fetch(f'SELECT DISTINCT tag_value.tag::text AS tag FROM {DB_SCHEMA}.visual_assets {value_source} WHERE tag_value.tag IS NOT NULL')
    _TAG_VOCABULARY_CACHE = _build_tag_vocabulary(rows)
    print(f"[Tag scope] Cached {len(_TAG_VOCABULARY_CACHE['entries']):,} canonical database tags locally; none were sent to Gemini.")
    return _TAG_VOCABULARY_CACHE

async def _load_hint_matched_tag_vocabulary(hints: list) -> dict:
    '''Try exact/synonym tag matches first so common Stage 1 runs do not enumerate the full 69k catalogue.'''
    terms = _expanded_tag_hint_terms(hints)
    if not terms:
        return {'entries': [], 'exact': {}, 'token_index': defaultdict(list)}
    column_types = await _get_visual_asset_column_types()
    native_tag_type = _native_tag_array_type(column_types)
    value_source = _tag_value_source(native_tag_type)
    tag_predicate = _tag_filter_predicate(native_tag_type, 1)
    if not value_source or not tag_predicate:
        return {'entries': [], 'exact': {}, 'token_index': defaultdict(list)}
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        rows = await db.fetch(
            f'''SELECT DISTINCT tag_value.tag::text AS tag
                FROM {DB_SCHEMA}.visual_assets
                {value_source}
                WHERE {tag_predicate}
                  AND tag_value.tag::text = ANY(CAST($1 AS text[]))''',
            terms,
        )
    return _build_tag_vocabulary(rows)

def _resolve_tag_scope(audit_config: dict, vocabulary: dict) -> tuple:
    hints = _tag_hint_terms(audit_config)
    if not hints or not vocabulary:
        return [], hints
    expanded_terms, excluded_terms, candidate_indexes = set(), set(), set()
    for hint in hints:
        expanded_terms.update(hint.split())
        for token in hint.split():
            expanded_terms.update(_TAG_SYNONYMS.get(token, {token}))
            excluded_terms.update(_TAG_CONFLICTS.get(token, set()))
        exact_index = vocabulary['exact'].get(hint)
        if exact_index is not None:
            candidate_indexes.add(exact_index)
    for term in expanded_terms:
        candidate_indexes.update(vocabulary['token_index'].get(term, []))
    ranked = []
    for index in candidate_indexes:
        entry = vocabulary['entries'][index]
        if entry['tokens'].intersection(excluded_terms):
            continue
        score = 0.0
        for hint in hints:
            hint_tokens = set(hint.split())
            if entry['normalised'] == hint:
                score += 24.0
            elif hint_tokens and hint_tokens.issubset(entry['tokens']):
                score += 10.0
            score += 3.0 * len(entry['tokens'].intersection(hint_tokens))
        score += len(entry['tokens'].intersection(expanded_terms))
        if entry['tokens'].issubset(TAG_SCOPE_TUNING['generic_tokens']):
            score -= 6.0
        if score > 0:
            ranked.append((score, entry['tag'], entry['tokens'].issubset(TAG_SCOPE_TUNING['generic_tokens'])))
    ranked.sort(key=lambda item: (-item[0], _normalise_tag_text(item[1])))
    if any(not is_generic for _, _, is_generic in ranked):
        ranked = [item for item in ranked if not item[2]]
    resolved = []
    for _, tag, _ in ranked:
        if tag not in resolved:
            resolved.append(tag)
        if len(resolved) >= TAG_SCOPE_TUNING['max_canonical_tags']:
            break
    return resolved, hints

def get_gcs_image_base64(gcs_path: str) -> str:
    """Downloads image from GCS or local file and returns its base64 data URI for inline HTML rendering."""
    try:
        image_bytes = b""
        mime_type = "image/png"

        # Detect format
        lower_path = gcs_path.lower()
        if lower_path.endswith(".jpg") or lower_path.endswith(".jpeg"):
            mime_type = "image/jpeg"
        elif lower_path.endswith(".webp"):
            mime_type = "image/webp"
        elif lower_path.endswith(".gif"):
            mime_type = "image/gif"

        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            bucket_name = parts[0]
            blob_name = parts[1]

            # Authenticated download using python client
            storage_client = storage.Client()
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            image_bytes = blob.download_as_bytes()
        elif os.path.exists(gcs_path):
            with open(gcs_path, "rb") as f:
                image_bytes = f.read()
        else:
            return ""

        encoded = base64.b64encode(image_bytes).decode("utf-8")
        return f"data:{mime_type};base64,{encoded}"
    except Exception as e:
        return ""

async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> dict:
    """Stage 1: Analyzes reference image (Forensic) and generates structured Audit Configuration."""
    reference_image_description = None
    if reference_image_path:
        print("0. Performing forensic executive analysis on reference image...")
        ref_mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            image_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=ref_mime)
        else:
            with open(reference_image_path, "rb") as f:
                image_part = types.Part.from_bytes(data=f.read(), mime_type=ref_mime)

        forensic_analysis_prompt = """You are a Lead Executive Visual & UI/UX Inspector. Perform an exhaustive, forensic-level breakdown of this uploaded reference image for enterprise audit configuration.
Provide a comprehensive, structured analysis starting with Brand Hierarchy:
1. Target Brand & Scope: What exact brand or product is shown? (e.g. specifically Google Pay / G Pay, or Google Workspace). Do not mix in separate products.
2. Primary Asset Hierarchy & Category: Classify whether this image is a Standalone Brand Logo, a UI Component (Payment Button, Cookie Modal), or a Composite Canvas (Hero Banner, Screenshot, Collage).
3. Exact Visual Signatures: Detail exact wordmarks, typography, overlapping geometric shapes (e.g. interlocking loops vs text wordmark), border radius, padding, and layout structure.
4. Color & Contrast Styling: Detail exact colors, contrast levels, and shadow/elevation effects.
5. Compliance & Style Classification: Only classify an asset as current or legacy when the user goal or visible evidence provides a reliable basis. Otherwise record the visual version signatures without guessing.
Do not apply one brand's version convention to another brand, product, person, or photograph. If, and only if, the reference is unambiguously Google Pay or G Pay, describe the visible logo form rather than inferring status from color alone."""

        resp = client.models.generate_content(
            model=GEMINI_ORCHESTRATOR_MODEL,
            contents=[image_part, forensic_analysis_prompt],
            config=types.GenerateContentConfig(temperature=0.0)
        )
        reference_image_description = resp.text
        print(f"Forensic Reference Image Analysis:\n{reference_image_description}\n{'='*50}")

    print("1. Translating goal into Audit Configuration...")
    # Generate a compact semantic configuration first. Passing [] prevents the 69k-tag catalogue from entering the model context.
    audit_config = generate_audit_config(user_goal, reference_image_description, available_tags=[])
    model_strategy = str(audit_config.get('evaluation_strategy', '')).strip().casefold()
    exact_hash_shortcut_authorized = _explicit_exact_bytes_mode(user_goal)
    audit_config['model_evaluation_strategy'] = model_strategy
    audit_config['exact_hash_shortcut_authorized'] = exact_hash_shortcut_authorized
    if exact_hash_shortcut_authorized:
        audit_config['evaluation_strategy'] = 'content_hash_exact'
    elif model_strategy == 'content_hash_exact':
        # A reference image alone is not authority to treat visually similar/cropped copies as byte-identical.
        audit_config['evaluation_strategy'] = 'visual_reference_variant' if reference_image_path else 'semantic_visual_presence'
    generated_tag_hints = list(audit_config.get('vision_tag_filter', []))
    try:
        tag_hints = _tag_hint_terms(audit_config)
        vocabulary = await _load_hint_matched_tag_vocabulary(tag_hints)
        resolved_tags, tag_hints = _resolve_tag_scope(audit_config, vocabulary)
        if not resolved_tags and tag_hints:
            # Do not enumerate all 69k tags from a 9.9M-row table merely to guess an unusual synonym.
            # Stage 2 fails open to the bounded global lane, which is safer than a costly lexical full-catalogue scan.
            print('[Tag scope] No exact/synonym canonical tag was found; skipping the full-catalogue fallback and retaining global-recall coverage.')
        audit_config['vision_tag_hints'] = generated_tag_hints
        audit_config['vision_tag_filter'] = resolved_tags
        audit_config['retrieval_tag_scope'] = resolved_tags
        audit_config['tag_scope_resolution'] = {
            'mode': 'local_catalogue_resolution',
            'hints': tag_hints,
            'resolved_tag_count': len(resolved_tags),
        }
        print(f"[Tag scope] Resolved {len(tag_hints)} model hints to {len(resolved_tags)} canonical tags for the scoped retrieval lane.")
        if not resolved_tags:
            print('[Tag scope] No safe canonical tags resolved; Stage 2 will fail open to the bounded global recall lane.')
    except Exception as e:
        audit_config['vision_tag_hints'] = generated_tag_hints
        audit_config['vision_tag_filter'] = []
        audit_config['retrieval_tag_scope'] = []
        audit_config['tag_scope_resolution'] = {'mode': 'unavailable_fail_open', 'error': str(e)}
        print(f'[Tag scope] Local resolution unavailable ({e}); Stage 2 will use the bounded global recall lane.')
    print("Generated Configuration:\n", json.dumps(audit_config, indent=2))
    return audit_config

async def run_full_test_bench_pipeline_execution(audit_config: dict, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Stage 2, 3, 3.5, 4, 5: Executes Retrieval, Semantic Reranking, Inference, and Calibration scorecards."""
    pipeline_start_time = time.time()
    telemetry = {}

    t0 = time.time()
    print("\n2. Running Content-Hash-Deduped Multimodal Search with RRF...")
    search_scope = {'max_unique_candidates': SEARCH_TUNING['quick_max_unique_candidates']} if quick_mode else {}
    search_results = await run_hybrid_search(search_scope, audit_config, reference_image_path)
    telemetry["Stage 2 (Hybrid RRF Retrieval)"] = f"{time.time() - t0:.2f}s | {len(search_results)} candidates retrieved"
    print(f"Found {len(search_results)} candidates.")

    t0 = time.time()
    print("\n3. Running Robust Two-Boundary Drop-off Analysis...")
    df_results = pd.DataFrame(search_results)
    df_high, df_edge, df_low = detect_dropoff_flawless(df_results)
    # Low is a ranked/deferred queue, never an implicit false verdict. Keep it available for a later recall sweep.
    globals()['deferred_dropoff_candidates_global'] = df_low.copy()
    telemetry["Stage 3 (Drop-off Segmentation)"] = f"{time.time() - t0:.2f}s | High: {len(df_high)}, Borderline: {len(df_edge)}, Low (Deferred ranked tail): {len(df_low)}"
    print(f"Rough Candidates -> High: {len(df_high)} | Borderline: {len(df_edge)} | Low: {len(df_low)}")

    t0 = time.time()
    print("\n3.5. Running Fast Contextual Reranking...")
    df_high_sem, df_edge_sem, df_low_sem = await run_semantic_reranking_and_filter(df_high, df_edge, audit_config)
    globals()['deferred_contextual_candidates_global'] = df_low_sem.copy()
    telemetry["Stage 3.5 (Semantic Reranking)"] = f"{time.time() - t0:.2f}s | High: {len(df_high_sem)}, Borderline: {len(df_edge_sem)}"
    print(f"Contextual Candidates -> High: {len(df_high_sem)} | Borderline: {len(df_edge_sem)} | Low (Not sent to visual audit): {len(df_low_sem)}")

    # Save full results globally for evaluation recall calculation
    globals()["df_results_full"] = df_results

    # Apply Quick Mode vs Smart Scan selection
    if quick_mode:
        print("\n Quick Mode enabled: Selecting top 30 High confidence and top 30 Borderline for visual inference.")
        candidates_high = df_high_sem.head(30) if not df_high_sem.empty else pd.DataFrame()
        candidates_edge = df_edge_sem.head(30) if not df_edge_sem.empty else pd.DataFrame()
    else:
        print("\n Smart Scan enabled: Passing all High and Borderline hashes to the diversity-aware visual-audit budget.")
        candidates_high = df_high_sem
        candidates_edge = df_edge_sem

    t0 = time.time()
    print("\n4. Running Parallel LLM Audit Inference...")
    results_df = await run_llm_inference_on_dropoff_results(candidates_high, candidates_edge, audit_config, reference_image_path=reference_image_path)
    instance_summary_df = await summarize_audited_instances(results_df)
    telemetry['Audited content-hash instance expansion'] = f'{len(instance_summary_df):,} audited hashes | {int(instance_summary_df["instance_count"].sum()) if not instance_summary_df.empty else 0:,} physical rows with propagated Boolean verdicts (paged)'
    inf_duration = time.time() - t0
    throughput = len(results_df) / inf_duration if inf_duration > 0 else 0
    telemetry["Stage 4 (Parallel Visual Inference)"] = f"{inf_duration:.2f}s | Throughput: {throughput:.2f} images/sec"
    print(f"LLM Results: {len(results_df)} assets audited.")

    try:
        from IPython.display import display, Markdown, HTML
        if not results_df.empty:
            display(Markdown('### Results'))
            preview_limit = SEARCH_TUNING['preview_rows']
            display_df = results_df.head(preview_limit).copy()
            def preview_html(path):
                preview = get_gcs_preview_base64(path)
                return f"<img src='{preview}' width='150' />" if preview else '[No Preview]'
            display_df['Visual Preview'] = display_df['gcs_raw_path'].apply(preview_html)
            display_df['Page Link'] = display_df['page_url'].apply(lambda value: f"<a href='{value}' target='_blank'>{value}</a>" if value else '[No Page Link]')
            columns = ['Visual Preview', 'matches_criteria', 'evaluation_status', 'confidence_band', 'requires_review', 'relevance_score', 'gcs_raw_path', 'Page Link', 'match_rationale']
            columns = [column for column in columns if column in display_df.columns]
            display(HTML(display_df[columns].to_html(escape=False, index=False)))
    except (ImportError, ModuleNotFoundError):
        if not results_df.empty:
            print(results_df.to_string(index=False))

    return results_df

# Bypassed original monolithic E2E pipeline name to map to partitioned runners
async def run_full_test_bench_pipeline(user_goal: str, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Wrapper to maintain backwards compatibility for existing cells."""
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode=quick_mode)


## E2E Execution & Telemetry Helpers

This cell defines the core pipeline orchestrators: `generate_audit_config_only` (Stage 1 Config) and `run_full_test_bench_pipeline_execution` (Stage 2 E2E). It coordinates retrieval, segmentation, cross-encoder reranking, and visual LLM inference, while logging telemetry and formatted scorecard widgets.

## Hybrid Search Recall, Precision, Deduplication & 3 GB Calibration

This is the active Stage 2 hybrid-search calibration layer. Run it once after the existing setup, configuration, connection, and base engine cells, and before the Stage 2 test cell. It does not authenticate, create clients, modify connection settings, or generate Stage 1 configuration. Stage 1 resolves a few semantic tag hints through native-array exact/synonym matches—never in a Gemini prompt and never by enumerating the full 69k-tag catalogue. Stage 2 uses those canonical tags to gate its primary vector, text, and tag arms only when `vision_tags` is a supported native text array; otherwise it safely fails open to its bounded global rescue lane. Each retrieval arm collapses physical duplicates to one representative per `content_hash` inside AlloyDB before returning rows to the notebook. After a representative passes, use the supplied paging helper to enumerate every physical asset instance without loading all 9.9 million rows into memory.

In [ ]:
# Active Stage 2 hybrid-search calibration. Run once after the base engine cells and before Stage 2; Stage 1 does not depend on this cell.
import hashlib
import re
from collections import defaultdict
from PIL import ImageOps

# These are test-bench budgets, not result caps. Increase them only after measuring query and model throughput.
SEARCH_TUNING = {
    'max_unique_candidates': 3000,
    'quick_max_unique_candidates': 600,
    'global_recall_unique_candidates': 500,
    # ~9.9M rows / ~500k content hashes means each hash has ~20 physical instances on average.
    # Probe beyond that duplicate ratio before the in-memory per-hash collapse, but cap it well below the 3 GB notebook budget.
    'per_arm_oversample': 24,
    'per_arm_fetch_cap': 72000,
    'per_arm_recall_reserve': {'reference_vector': 240, 'text_vector': 160, 'fts': 160, 'tags': 160},
    'max_cross_encoder_candidates': 360,
    'max_visual_audit_candidates': 240,
    'cross_encoder_workers': 12,
    'inference_workers': 12,
    'inference_batch_size': 24,
    'instance_page_size': 500,
    'preview_rows': 25,
    'max_source_image_bytes': 24 * 1024 * 1024,
    'max_model_image_pixels': 4_000_000,
    'visual_safeguard_distance': 0.28,
    'minimum_review_confidence': 60,
}
_AUDIT_RESULT_CACHE = {}

def _content_key(row: dict) -> str:
    return str(row.get('content_hash') or row.get('gcs_raw_path') or row.get('asset_id'))

def _as_float(value, default=1.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def _unique_by_content_hash(rows: list, ranking_key: str = None, descending: bool = False) -> list:
    ordered = rows
    if ranking_key:
        ordered = sorted(rows, key=lambda row: _as_float(row.get(ranking_key)), reverse=descending)
    seen = set()
    unique = []
    for row in ordered:
        key = _content_key(row)
        if key not in seen:
            seen.add(key)
            unique.append(row)
    return unique

def _search_terms(audit_context: dict) -> list:
    '''Create safe lexical tokens; never collapse a phrase such as 'Google Pay' into the non-word 'googlepay'.'''
    terms = []
    for raw_term in audit_context.get('search_keywords', []):
        for term in re.findall(r'[a-z0-9]+', str(raw_term).casefold()):
            if len(term) >= 2 and term not in terms:
                terms.append(term)
    return terms[:16]

def _exact_tags(audit_context: dict) -> list:
    resolved_scope = audit_context.get('retrieval_tag_scope') or audit_context.get('vision_tag_filter', [])
    return list(dict.fromkeys(str(tag).strip() for tag in resolved_scope if str(tag).strip()))

def _read_image_bytes(path: str, max_bytes: int = None) -> bytes:
    byte_limit = max_bytes or SEARCH_TUNING['max_source_image_bytes']
    if path.startswith('gs://'):
        bucket_name, blob_name = path[5:].split('/', 1)
        storage_client = storage.Client(project=PROJECT_ID)
        blob = storage_client.bucket(bucket_name).blob(blob_name)
        blob.reload()
        if blob.size is not None and blob.size > byte_limit:
            raise ValueError(f'Image is {blob.size:,} bytes; limit is {byte_limit:,} bytes')
        return blob.download_as_bytes()
    if os.path.getsize(path) > byte_limit:
        raise ValueError(f'Image exceeds the {byte_limit:,}-byte model input limit')
    with open(path, 'rb') as image_file:
        return image_file.read()

def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    '''Normalise orientation, alpha and dimensions, then always emit a correctly labelled PNG.'''
    if len(image_bytes) > SEARCH_TUNING['max_source_image_bytes']:
        raise ValueError('Image exceeds the configured model-input byte budget')
    with Image.open(io.BytesIO(image_bytes)) as opened:
        image = ImageOps.exif_transpose(opened)
        max_side = int(SEARCH_TUNING['max_model_image_pixels'] ** 0.5)
        image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        has_alpha = image.mode in ('RGBA', 'LA') or (image.mode == 'P' and 'transparency' in image.info)
        if has_alpha:
            foreground = image.convert('RGBA')
            background = Image.new('RGBA', foreground.size, default_bg + (255,))
            image = Image.alpha_composite(background, foreground).convert('RGB')
        else:
            image = image.convert('RGB')
        output = io.BytesIO()
        image.save(output, format='PNG', optimize=True)
        return output.getvalue()

def get_gcs_preview_base64(gcs_path: str, max_side: int = 256) -> str:
    '''Build one small preview at a time; never embed original image bytes in the result table.'''
    try:
        with Image.open(io.BytesIO(_read_image_bytes(gcs_path))) as opened:
            image = ImageOps.exif_transpose(opened).convert('RGB')
            image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
            output = io.BytesIO()
            image.save(output, format='JPEG', quality=80, optimize=True)
        return 'data:image/jpeg;base64,' + base64.b64encode(output.getvalue()).decode('utf-8')
    except Exception:
        return ''

def _reference_hashes(reference_image_path: Optional[str]) -> list:
    if not reference_image_path:
        return []
    raw_bytes = _read_image_bytes(reference_image_path, max_bytes=SEARCH_TUNING['max_source_image_bytes'])
    return [hashlib.sha256(raw_bytes).hexdigest(), hashlib.md5(raw_bytes).hexdigest()]

async def _exact_content_hash_matches(reference_image_path: Optional[str]) -> list:
    hashes = _reference_hashes(reference_image_path)
    if not hashes:
        return []
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        rows = await db.fetch(
            f'''
            SELECT content_hash, MIN(asset_id::text) AS asset_id, MIN(gcs_raw_path) AS gcs_raw_path,
                   MIN(format::text) AS format, MIN(asset_filename) AS asset_filename, MIN(page_url) AS page_url,
                   COUNT(*) AS exact_instance_count
            FROM {DB_SCHEMA}.visual_assets
            WHERE content_hash = ANY($1::text[])
            GROUP BY content_hash
            ''',
            hashes,
        )
    matches = []
    for row in rows:
        item = dict(row)
        item.update({'exact_content_match': True, 'vector_distance': 0.0, 'retrieval_sources': 'exact_hash'})
        matches.append(item)
    return matches

async def embed_audit_context(audit_context: dict, reference_image_path: Optional[str] = None) -> dict:
    '''Keep visual reference and text intent as separate retrieval signals instead of diluting either one.'''
    positive_text = build_fused_query_text(audit_context, is_negative=False)[:1000]
    negative_text = build_fused_query_text(audit_context, is_negative=True)[:1000]
    reference_part = None
    if reference_image_path:
        reference_part = types.Part.from_bytes(data=_read_image_bytes(reference_image_path), mime_type=get_image_mime_type(reference_image_path))

    def _embed(contents):
        return client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=contents,
            config=types.EmbedContentConfig(output_dimensionality=768),
        ).embeddings[0].values

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=3) as executor:
        text_future = loop.run_in_executor(executor, _embed, [positive_text])
        negative_future = loop.run_in_executor(executor, _embed, [negative_text])
        reference_future = loop.run_in_executor(executor, _embed, [reference_part]) if reference_part else None
        text_vector, negative_vector = await asyncio.gather(text_future, negative_future)
        reference_vector = await reference_future if reference_future else None
    return {'text': text_vector, 'reference': reference_vector, 'negative': negative_vector}

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: Optional[int] = None) -> List[dict]:
    '''Recall-first hybrid search where each arm is deduplicated by content hash before RRF.'''
    max_unique = int(scope_config.get('max_unique_candidates', limit or SEARCH_TUNING['max_unique_candidates']))
    fetch_limit = min(max_unique * SEARCH_TUNING['per_arm_oversample'], SEARCH_TUNING['per_arm_fetch_cap'])
    global_recall_unique = min(max_unique, SEARCH_TUNING['global_recall_unique_candidates'])
    global_fetch_limit = min(global_recall_unique * SEARCH_TUNING['per_arm_oversample'], SEARCH_TUNING['per_arm_fetch_cap'])
    vectors = await embed_audit_context(audit_context, reference_image_path)
    terms = _search_terms(audit_context)
    tags = _exact_tags(audit_context)
    tag_scope = list(dict.fromkeys(str(tag).strip() for tag in audit_context.get('retrieval_tag_scope', tags) if str(tag).strip()))
    requested_tag_scope_count = len(tag_scope)
    keyword_query = ' OR '.join(terms)
    engine, _ = await get_alloydb_connection()
    arm_errors = []
    try:
        column_types = await _get_visual_asset_column_types()
        native_tag_type = _native_tag_array_type(column_types)
    except Exception as error:
        column_types, native_tag_type = {}, None
        arm_errors.append({'arm': 'tag-schema-introspection', 'error': str(error)})
        print(f'[Tag scope] Could not read vision_tags type ({error}); using global retrieval without tag filtering.')
    if tag_scope and not native_tag_type:
        print(f"[Tag scope] vision_tags type is {column_types.get('vision_tags', 'unknown')!r}, not a supported native text array; using global retrieval without tag filtering.")
        tag_scope = []

    async def vector_arm(vector, distance_name, tag_filter=None, arm_limit=None):
        if vector is None:
            return []
        arm_limit = arm_limit or fetch_limit
        unique_limit = max(1, arm_limit // SEARCH_TUNING['per_arm_oversample'])
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            if tag_filter:
                tag_predicate = _tag_filter_predicate(native_tag_type, 3)
                rows = await db.fetch(
                    f'''WITH nearest_physical AS (
                            SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                                   COALESCE(NULLIF(content_hash::text, ''), NULLIF(gcs_raw_path::text, ''), asset_id::text) AS content_key,
                                   (embedding <=> $1::vector) AS {distance_name}_distance,
                                   (embedding <=> $2::vector) AS negative_vector_distance
                            FROM {DB_SCHEMA}.visual_assets
                            WHERE embedding IS NOT NULL AND {tag_predicate}
                            ORDER BY embedding <=> $1::vector
                            LIMIT $4
                        ), nearest_unique AS (
                            SELECT DISTINCT ON (content_key) * FROM nearest_physical
                            ORDER BY content_key, {distance_name}_distance ASC
                        )
                        SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash, {distance_name}_distance, negative_vector_distance
                        FROM nearest_unique ORDER BY {distance_name}_distance ASC LIMIT $5''',
                    vector, vectors['negative'], tag_filter, arm_limit, unique_limit,
                )
            else:
                rows = await db.fetch(
                    f'''WITH nearest_physical AS (
                            SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                                   COALESCE(NULLIF(content_hash::text, ''), NULLIF(gcs_raw_path::text, ''), asset_id::text) AS content_key,
                                   (embedding <=> $1::vector) AS {distance_name}_distance,
                                   (embedding <=> $2::vector) AS negative_vector_distance
                            FROM {DB_SCHEMA}.visual_assets
                            WHERE embedding IS NOT NULL
                            ORDER BY embedding <=> $1::vector
                            LIMIT $3
                        ), nearest_unique AS (
                            SELECT DISTINCT ON (content_key) * FROM nearest_physical
                            ORDER BY content_key, {distance_name}_distance ASC
                        )
                        SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash, {distance_name}_distance, negative_vector_distance
                        FROM nearest_unique ORDER BY {distance_name}_distance ASC LIMIT $4''',
                    vector, vectors['negative'], arm_limit, unique_limit,
                )
        return [dict(row) for row in rows]

    async def fts_arm(tag_filter=None, arm_limit=None):
        if not keyword_query:
            return []
        arm_limit = arm_limit or fetch_limit
        unique_limit = max(1, arm_limit // SEARCH_TUNING['per_arm_oversample'])
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            if tag_filter:
                tag_predicate = _tag_filter_predicate(native_tag_type, 2)
                rows = await db.fetch(
                    f'''WITH query AS (SELECT websearch_to_tsquery('english', $1) AS tsq),
                        ranked_physical AS (
                            SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                                   COALESCE(NULLIF(content_hash::text, ''), NULLIF(gcs_raw_path::text, ''), asset_id::text) AS content_key,
                                   ts_rank_cd(to_tsvector('english', concat_ws(' ', coalesce(gemini_description::text, ''), coalesce(asset_filename::text, ''))), query.tsq) AS fts_rank
                            FROM {DB_SCHEMA}.visual_assets, query
                            WHERE {tag_predicate} AND to_tsvector('english', concat_ws(' ', coalesce(gemini_description::text, ''), coalesce(asset_filename::text, ''))) @@ query.tsq
                            ORDER BY fts_rank DESC, asset_id
                            LIMIT $3
                        ), ranked_unique AS (
                            SELECT DISTINCT ON (content_key) * FROM ranked_physical
                            ORDER BY content_key, fts_rank DESC, asset_id
                        )
                        SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash, fts_rank
                        FROM ranked_unique ORDER BY fts_rank DESC, asset_id LIMIT $4''',
                    keyword_query, tag_filter, arm_limit, unique_limit,
                )
            else:
                rows = await db.fetch(
                    f'''WITH query AS (SELECT websearch_to_tsquery('english', $1) AS tsq),
                        ranked_physical AS (
                            SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                                   COALESCE(NULLIF(content_hash::text, ''), NULLIF(gcs_raw_path::text, ''), asset_id::text) AS content_key,
                                   ts_rank_cd(to_tsvector('english', concat_ws(' ', coalesce(gemini_description::text, ''), coalesce(asset_filename::text, ''))), query.tsq) AS fts_rank
                            FROM {DB_SCHEMA}.visual_assets, query
                            WHERE to_tsvector('english', concat_ws(' ', coalesce(gemini_description::text, ''), coalesce(asset_filename::text, ''))) @@ query.tsq
                            ORDER BY fts_rank DESC, asset_id
                            LIMIT $2
                        ), ranked_unique AS (
                            SELECT DISTINCT ON (content_key) * FROM ranked_physical
                            ORDER BY content_key, fts_rank DESC, asset_id
                        )
                        SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash, fts_rank
                        FROM ranked_unique ORDER BY fts_rank DESC, asset_id LIMIT $3''',
                    keyword_query, arm_limit, unique_limit,
                )
        return [dict(row) for row in rows]

    async def tag_arm():
        if not tags or not native_tag_type:
            return []
        unique_limit = max(1, fetch_limit // SEARCH_TUNING['per_arm_oversample'])
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f'''WITH tagged_physical AS (
                    SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                           COALESCE(NULLIF(content_hash::text, ''), NULLIF(gcs_raw_path::text, ''), asset_id::text) AS content_key,
                           cardinality(ARRAY(SELECT tag_value.tag::text FROM unnest(vision_tags) AS tag_value(tag) WHERE tag_value.tag::text = ANY(CAST($1 AS text[])))) AS tag_match_count
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE {_tag_filter_predicate(native_tag_type, 1)}
                    ORDER BY tag_match_count DESC, asset_id
                    LIMIT $2
                ), tagged_unique AS (
                    SELECT DISTINCT ON (content_key) * FROM tagged_physical
                    ORDER BY content_key, tag_match_count DESC, asset_id
                )
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash, tag_match_count
                FROM tagged_unique ORDER BY tag_match_count DESC, asset_id LIMIT $3''',
                tags, fetch_limit, unique_limit,
            )
        return [dict(row) for row in rows]

    async def safe_arm(name, coroutine):
        try:
            return await coroutine
        except Exception as error:
            arm_errors.append({'arm': name, 'error': str(error)})
            print(f'[Retrieval warning] {name} arm failed: {error}')
            return []

    if tag_scope:
        scoped_reference_rows, scoped_text_rows, scoped_fts_rows, reference_rows, text_rows, fts_rows, tag_rows, exact_rows = await asyncio.gather(
            safe_arm('scoped-reference-vector', vector_arm(vectors['reference'], 'reference_vector', tag_scope, fetch_limit)),
            safe_arm('scoped-text-vector', vector_arm(vectors['text'], 'text_vector', tag_scope, fetch_limit)),
            safe_arm('scoped-full-text', fts_arm(tag_scope, fetch_limit)),
            safe_arm('global-reference-rescue', vector_arm(vectors['reference'], 'reference_vector', None, global_fetch_limit)),
            safe_arm('global-text-rescue', vector_arm(vectors['text'], 'text_vector', None, global_fetch_limit)),
            safe_arm('global-full-text-rescue', fts_arm(None, global_fetch_limit)),
            safe_arm('vision-tag', tag_arm()),
            safe_arm('exact-content-hash', _exact_content_hash_matches(reference_image_path)),
        )
        arm_specs = []
        if scoped_reference_rows:
            arm_specs.append(('reference_vector_scoped', _unique_by_content_hash(scoped_reference_rows, 'reference_vector_distance'), 0.50))
        if scoped_text_rows:
            arm_specs.append(('text_vector_scoped', _unique_by_content_hash(scoped_text_rows, 'text_vector_distance'), 0.38 if scoped_reference_rows else 0.55))
        if scoped_fts_rows:
            arm_specs.append(('fts_scoped', _unique_by_content_hash(scoped_fts_rows, 'fts_rank', descending=True), 0.28))
        if reference_rows:
            arm_specs.append(('reference_vector', _unique_by_content_hash(reference_rows, 'reference_vector_distance'), 0.20))
        if text_rows:
            arm_specs.append(('text_vector', _unique_by_content_hash(text_rows, 'text_vector_distance'), 0.16 if reference_rows else 0.30))
        if fts_rows:
            arm_specs.append(('fts', _unique_by_content_hash(fts_rows, 'fts_rank', descending=True), 0.12))
        arm_specs.extend([
            ('tags', _unique_by_content_hash(tag_rows, 'tag_match_count', descending=True), 0.18),
            ('exact_hash', _unique_by_content_hash(exact_rows), 1.00),
        ])
        print(f'[Tag scope] {len(tag_scope):,} canonical tags gate the primary retrieval lane; each global rescue arm is limited to {global_recall_unique:,} unique-candidate equivalents.')
    else:
        reference_rows, text_rows, fts_rows, tag_rows, exact_rows = await asyncio.gather(
            safe_arm('reference-vector', vector_arm(vectors['reference'], 'reference_vector')),
            safe_arm('text-vector', vector_arm(vectors['text'], 'text_vector')),
            safe_arm('full-text', fts_arm()),
            safe_arm('vision-tag', tag_arm()),
            safe_arm('exact-content-hash', _exact_content_hash_matches(reference_image_path)),
        )
        arm_specs = []
        if reference_rows:
            arm_specs.append(('reference_vector', _unique_by_content_hash(reference_rows, 'reference_vector_distance'), 0.40))
        arm_specs.append(('text_vector', _unique_by_content_hash(text_rows, 'text_vector_distance'), 0.30 if reference_rows else 0.60))
        arm_specs.extend([
            ('fts', _unique_by_content_hash(fts_rows, 'fts_rank', descending=True), 0.20 if reference_rows else 0.25),
            ('tags', _unique_by_content_hash(tag_rows, 'tag_match_count', descending=True), 0.10 if reference_rows else 0.15),
            ('exact_hash', _unique_by_content_hash(exact_rows), 1.00),
        ])
    fused = {}
    rrf_k = 60
    for source_name, rows, weight in arm_specs:
        for rank, row in enumerate(rows):
            key = _content_key(row)
            entry = fused.setdefault(key, {'data': dict(row), 'score': 0.0, 'sources': set(), 'arm_ranks': {}})
            entry['score'] += weight / (rrf_k + rank + 1)
            entry['sources'].add(source_name)
            prior_rank = entry['arm_ranks'].get(source_name)
            entry['arm_ranks'][source_name] = min(prior_rank, rank + 1) if prior_rank is not None else rank + 1
            for distance_name in ('reference_vector_distance', 'text_vector_distance', 'negative_vector_distance'):
                if row.get(distance_name) is not None:
                    existing = entry['data'].get(distance_name)
                    if existing is None or _as_float(row[distance_name]) < _as_float(existing):
                        entry['data'][distance_name] = row[distance_name]
            if row.get('exact_content_match'):
                entry['data'].update(row)

    globals()['hybrid_retrieval_diagnostics_global'] = {
        'failed_arms': arm_errors,
        'requested_tag_scope_count': requested_tag_scope_count,
        'active_tag_scope_count': len(tag_scope),
        'vision_tags_storage_type': column_types.get('vision_tags'),
        'primary_physical_fetch_limit': fetch_limit,
        'global_rescue_physical_fetch_limit': global_fetch_limit,
    }
    if not fused:
        # A valid no-match search must complete cleanly; failures remain explicit for diagnosis instead of being misreported as false matches.
        outcome = 'All enabled retrieval arms failed' if arm_errors else 'No candidates were returned by the available arms'
        print(f'[Retrieval] {outcome}. Stage 2 will complete with an empty result set.')
        return []
    candidates = []
    normalized_tags = {tag.casefold() for tag in tags}
    normalized_terms = set(terms)
    for entry in fused.values():
        row = entry['data']
        row_tags = {str(tag).casefold() for tag in (row.get('vision_tags') or [])}
        tag_hits = len(row_tags.intersection(normalized_tags))
        searchable_text = f"{row.get('gemini_description') or ''} {row.get('asset_filename') or ''}".casefold()
        keyword_hits = sum(1 for term in normalized_terms if term in searchable_text)
        negative_distance = _as_float(row.get('negative_vector_distance'), 1.0)
        negative_penalty = max(0.0, min(0.20, (0.35 - negative_distance) * 0.55))
        signal_multiplier = 1.0 + 0.18 * min(tag_hits, 2) + 0.08 * min(keyword_hits, 3) - negative_penalty
        reference_distance = _as_float(row.get('reference_vector_distance'), 9.0)
        text_distance = _as_float(row.get('text_vector_distance'), 9.0)
        # Never let a text-embedding distance masquerade as visual-reference evidence.
        best_embedding_distance = min(reference_distance, text_distance)
        vector_distance = reference_distance
        if row.get('exact_content_match'):
            relevance_score = 1.0
            vector_distance = 0.0
        else:
            relevance_score = max(0.0, entry['score'] * signal_multiplier)
        candidates.append({
            **row,
            'relevance_score': relevance_score,
            'base_rrf_score': entry['score'],
            'rerank_multiplier': signal_multiplier,
            'vector_distance': vector_distance,
            'visual_reference_distance': reference_distance if reference_distance < 9.0 else None,
            'best_embedding_distance': best_embedding_distance,
            'retrieval_sources': ','.join(sorted(entry['sources'])),
            'arm_ranks': dict(entry['arm_ranks']),
            'best_arm_rank': min(entry['arm_ranks'].values()) if entry['arm_ranks'] else None,
            'promoted_by_keyword_or_tag': bool({'fts', 'fts_scoped', 'tags'}.intersection(entry['sources'])),
        })
    candidates.sort(key=lambda row: row['relevance_score'], reverse=True)
    unique_arm_counts = {name: len(rows) for name, rows, _ in arm_specs}
    globals()['hybrid_retrieval_diagnostics_global'].update({'per_arm_unique_returned': unique_arm_counts, 'fused_unique_hashes': len(candidates)})
    print(f'[Retrieval] {len(candidates):,} fused unique content hashes after per-arm deduplication; per-arm unique counts: {unique_arm_counts}; fetched at most {fetch_limit:,} physical rows per arm.')
    return candidates[:max_unique]

def _select_diverse_candidates(candidates_df: pd.DataFrame, budget: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if candidates_df.empty or len(candidates_df) <= budget:
        return candidates_df.copy(), pd.DataFrame(columns=candidates_df.columns)
    ordered = candidates_df.sort_values('relevance_score', ascending=False).drop_duplicates(subset=['content_hash', 'gcs_raw_path'], keep='first').reset_index(drop=True)
    priority_mask = (
        ordered.get('exact_content_match', pd.Series(False, index=ordered.index)).fillna(False)
        | (pd.to_numeric(ordered.get('visual_reference_distance', pd.Series(9.0, index=ordered.index)), errors='coerce').fillna(9.0) <= SEARCH_TUNING['visual_safeguard_distance'])
    )
    selected_indices = []
    # Exact copies are deterministic positives for an exact-identity audit. Keep them first, then reserve room for every retrieval arm.
    for index in ordered.loc[ordered.get('exact_content_match', pd.Series(False, index=ordered.index)).fillna(False)].index:
        if len(selected_indices) >= budget:
            break
        selected_indices.append(index)
    def arm_rank(row, source):
        ranks = row.get('arm_ranks') or {}
        if not isinstance(ranks, dict):
            return float('inf')
        matching_ranks = [_as_float(rank, float('inf')) for arm, rank in ranks.items() if _source_family(arm) == source]
        return min(matching_ranks) if matching_ranks else float('inf')
    reserve_quota = max(1, min(30, budget // 8))
    for source, reserve in SEARCH_TUNING['per_arm_recall_reserve'].items():
        reserve_rows = [(arm_rank(row, source), index) for index, row in ordered.iterrows() if arm_rank(row, source) <= reserve]
        for _, index in sorted(reserve_rows):
            if len(selected_indices) >= budget:
                break
            if index not in selected_indices:
                selected_indices.append(index)
                if sum(1 for chosen in selected_indices if arm_rank(ordered.loc[chosen], source) <= reserve) >= reserve_quota:
                    break
    # Strong reference evidence remains important, but no longer consumes the entire visual-audit budget before text/tag reserves are represented.
    for index in ordered.loc[priority_mask].index:
        if len(selected_indices) >= budget:
            break
        if index not in selected_indices:
            selected_indices.append(index)
    per_source_quota = max(1, budget // 4)
    retrieval_sources = ordered.get('retrieval_sources', pd.Series('', index=ordered.index)).fillna('').astype(str)
    for source in ('reference_vector', 'text_vector', 'fts', 'tags'):
        source_rows = ordered[retrieval_sources.str.contains(source, regex=False)]
        for index in source_rows.index:
            if len(selected_indices) >= budget:
                break
            if index not in selected_indices:
                selected_indices.append(index)
                if sum(1 for chosen in selected_indices if source in str(ordered.loc[chosen].get('retrieval_sources', ''))) >= per_source_quota:
                    break
    for index in ordered.index:
        if len(selected_indices) >= budget:
            break
        if index not in selected_indices:
            selected_indices.append(index)
    selected = ordered.loc[selected_indices].sort_values('relevance_score', ascending=False).reset_index(drop=True)
    deferred = ordered.drop(index=selected_indices).copy()
    deferred['deferred_for_full_recall'] = True
    return selected, deferred.reset_index(drop=True)

# Final hybrid refinement: robust two-knee segmentation plus deterministic contextual reranking.
# It keeps the final image verdict binary and does not alter comprehensive scan behaviour.
def _source_family(source: str) -> str:
    return str(source).removesuffix('_scoped')

def _source_names(row: dict) -> set:
    return {_source_family(source) for source in str(row.get('retrieval_sources') or '').split(',') if source}

def _robust_hybrid_boundaries(scores: np.ndarray) -> Tuple[int, int, np.ndarray]:
    '''Return High and High+Borderline row counts from sustained score gaps, with safe rank-based fallbacks.'''
    count = len(scores)
    if count <= 2:
        return count, count, np.zeros(max(0, count - 1))
    positive_scores = np.maximum(np.asarray(scores, dtype=float), np.finfo(float).eps)
    log_scores = np.log(positive_scores)
    window = min(21, max(3, 2 * int(np.sqrt(count) // 4) + 1))
    smoothed = pd.Series(log_scores).rolling(window=window, center=True, min_periods=1).median().to_numpy()
    gaps = np.maximum(0.0, smoothed[:-1] - smoothed[1:])
    gap_median = float(np.median(gaps))
    gap_mad = float(np.median(np.abs(gaps - gap_median)))
    robust_z = (gaps - gap_median) / max(1.4826 * gap_mad, 1e-9)

    minimum_high = min(count - 1, max(5, int(np.ceil(count * 0.01))))
    maximum_high = min(count - 2, max(minimum_high, int(np.ceil(count * 0.35))))
    default_high = min(maximum_high, max(minimum_high, int(np.ceil(count * 0.10))))
    high_gap_range = np.arange(max(0, minimum_high - 1), max(0, maximum_high))
    if len(high_gap_range):
        high_gap = int(high_gap_range[np.argmax(robust_z[high_gap_range])])
        high_count = high_gap + 1 if robust_z[high_gap] >= 1.25 else default_high
    else:
        high_count = default_high

    minimum_borderline = min(count - 1, high_count + max(8, int(np.ceil(count * 0.03))))
    maximum_borderline = min(count - 1, max(minimum_borderline, int(np.ceil(count * 0.80))))
    default_borderline = min(maximum_borderline, max(minimum_borderline, int(np.ceil(count * 0.55))))
    borderline_gap_range = np.arange(max(0, minimum_borderline - 1), max(0, maximum_borderline))
    if len(borderline_gap_range):
        borderline_gap = int(borderline_gap_range[np.argmax(robust_z[borderline_gap_range])])
        borderline_count = borderline_gap + 1 if robust_z[borderline_gap] >= 0.75 else default_borderline
    else:
        borderline_count = default_borderline

    high_count = max(1, min(high_count, count - 1))
    borderline_count = max(high_count + 1, min(borderline_count, count))
    return high_count, borderline_count, robust_z

def _recall_reserve_sources(row: dict) -> list:
    '''Return retrieval arms in which this hash ranked strongly even if its fused score is modest.'''
    arm_ranks = row.get('arm_ranks') or {}
    if not isinstance(arm_ranks, dict):
        return []
    reserves = SEARCH_TUNING['per_arm_recall_reserve']
    retained = []
    for source, reserve in reserves.items():
        matching_ranks = [_as_float(rank, float('inf')) for arm, rank in arm_ranks.items() if _source_family(arm) == source]
        if matching_ranks and min(matching_ranks) <= reserve:
            retained.append(source)
    return retained

def _detect_dropoff_single_curve_legacy(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''Legacy single-curve segmentation retained only for comparison; tag-scoped and global-rescue scores are not calibrated together.'''
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    ranked = df.sort_values('relevance_score', ascending=False).drop_duplicates(subset=['content_hash', 'gcs_raw_path'], keep='first').reset_index(drop=True).copy()
    ranked['retrieval_rank'] = np.arange(1, len(ranked) + 1)
    ranked['retrieval_rank_percentile'] = 1.0 - ((ranked['retrieval_rank'] - 1) / max(1, len(ranked) - 1))
    if len(ranked) <= 2:
        ranked['dropoff_band'] = 'high'
        ranked['dropoff_reason'] = 'small_result_set'
        return ranked, pd.DataFrame(columns=ranked.columns), pd.DataFrame(columns=ranked.columns)

    high_count, borderline_count, gap_z = _robust_hybrid_boundaries(ranked['relevance_score'].to_numpy())
    ranked['dropoff_gap_z'] = np.append(gap_z, 0.0)
    ranked['dropoff_band'] = np.where(ranked.index < high_count, 'high', np.where(ranked.index < borderline_count, 'borderline', 'low'))
    ranked['dropoff_reason'] = np.where(ranked.index < high_count, 'first_sustained_gap', np.where(ranked.index < borderline_count, 'second_sustained_gap', 'tail_score'))

    source_counts = ranked.apply(lambda row: len(_source_names(row)), axis=1)
    source_text = ranked.get('retrieval_sources', pd.Series('', index=ranked.index)).fillna('').astype(str)
    exact_hash = ranked.get('exact_content_match', pd.Series(False, index=ranked.index)).fillna(False)
    reference_distance = pd.to_numeric(ranked.get('reference_vector_distance', pd.Series(9.0, index=ranked.index)), errors='coerce').fillna(9.0)
    has_reference_signal = source_text.str.contains('reference_vector', regex=False)
    strong_visual = has_reference_signal & (reference_distance <= SEARCH_TUNING['visual_safeguard_distance'])
    corroborated = source_counts >= 2
    ranked['recall_reserve_source'] = ranked.apply(lambda row: ','.join(_recall_reserve_sources(row)), axis=1)
    per_arm_recall_reserve = ranked['recall_reserve_source'].ne('')
    rescue_to_borderline = (ranked['dropoff_band'] == 'low') & (strong_visual | corroborated | per_arm_recall_reserve)
    ranked.loc[exact_hash, ['dropoff_band', 'dropoff_reason']] = ['high', 'exact_content_hash']
    rescue_reason = np.where(per_arm_recall_reserve, 'per_arm_recall_reserve', np.where(strong_visual, 'strong_reference_tail', 'corroborated_tail_signal'))
    ranked.loc[rescue_to_borderline & ~exact_hash, 'dropoff_band'] = 'borderline'
    ranked.loc[rescue_to_borderline & ~exact_hash, 'dropoff_reason'] = rescue_reason[rescue_to_borderline & ~exact_hash]
    ranked['rescued_for_recall'] = rescue_to_borderline | exact_hash

    high_df = ranked[ranked['dropoff_band'] == 'high'].copy()
    edge_df = ranked[ranked['dropoff_band'] == 'borderline'].copy()
    low_df = ranked[ranked['dropoff_band'] == 'low'].copy()
    print(f'[Drop-off] Two robust boundaries: High {len(high_df):,}, Borderline {len(edge_df):,}, Low {len(low_df):,}.')
    return high_df, edge_df, low_df

def _retrieval_lane(row: dict) -> str:
    '''Tag-scoped and global-rescue scores are ranked independently because their raw RRF values are not calibrated to one another.'''
    sources = {source for source in str(row.get('retrieval_sources') or '').split(',') if source}
    return 'tag_scoped' if any(source.endswith('_scoped') for source in sources) or 'tags' in sources else 'global_rescue'

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''Create High/Borderline queues per retrieval lane. Low is explicitly deferred, never treated as disproved.'''
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    ranked = df.sort_values('relevance_score', ascending=False).drop_duplicates(subset=['content_hash', 'gcs_raw_path'], keep='first').reset_index(drop=True).copy()
    ranked['overall_retrieval_rank'] = np.arange(1, len(ranked) + 1)
    ranked['retrieval_lane'] = ranked.apply(_retrieval_lane, axis=1)
    ranked['retrieval_rank'] = 0
    ranked['retrieval_rank_percentile'] = 0.0
    ranked['dropoff_gap_z'] = 0.0
    ranked['dropoff_band'] = 'low'
    ranked['dropoff_reason'] = 'lane_tail_score'

    for lane in ('tag_scoped', 'global_rescue'):
        lane_indices = ranked[ranked['retrieval_lane'] == lane].sort_values('relevance_score', ascending=False).index
        lane_count = len(lane_indices)
        if not lane_count:
            continue
        lane_ranks = np.arange(1, lane_count + 1)
        ranked.loc[lane_indices, 'retrieval_rank'] = lane_ranks
        ranked.loc[lane_indices, 'retrieval_rank_percentile'] = 1.0 - ((lane_ranks - 1) / max(1, lane_count - 1))
        if lane_count <= 2:
            ranked.loc[lane_indices, ['dropoff_band', 'dropoff_reason']] = ['high', f'{lane}_small_result_set']
            continue
        high_count, borderline_count, gap_z = _robust_hybrid_boundaries(ranked.loc[lane_indices, 'relevance_score'].to_numpy())
        ranked.loc[lane_indices, 'dropoff_gap_z'] = np.append(gap_z, 0.0)
        ranked.loc[lane_indices[:high_count], ['dropoff_band', 'dropoff_reason']] = ['high', f'{lane}_first_sustained_gap']
        ranked.loc[lane_indices[high_count:borderline_count], ['dropoff_band', 'dropoff_reason']] = ['borderline', f'{lane}_second_sustained_gap']

    source_counts = ranked.apply(lambda row: len(_source_names(row)), axis=1)
    source_text = ranked.get('retrieval_sources', pd.Series('', index=ranked.index)).fillna('').astype(str)
    exact_hash = ranked.get('exact_content_match', pd.Series(False, index=ranked.index)).fillna(False)
    reference_distance = pd.to_numeric(ranked.get('reference_vector_distance', pd.Series(9.0, index=ranked.index)), errors='coerce').fillna(9.0)
    strong_visual = source_text.str.contains('reference_vector', regex=False) & (reference_distance <= SEARCH_TUNING['visual_safeguard_distance'])
    corroborated = source_counts >= 2
    ranked['recall_reserve_source'] = ranked.apply(lambda row: ','.join(_recall_reserve_sources(row)), axis=1)
    per_arm_recall_reserve = ranked['recall_reserve_source'].ne('')
    rescue_to_borderline = (ranked['dropoff_band'] == 'low') & (strong_visual | corroborated | per_arm_recall_reserve)
    ranked.loc[exact_hash, ['dropoff_band', 'dropoff_reason']] = ['high', 'exact_content_hash']
    rescue_reason = np.where(per_arm_recall_reserve, 'per_arm_recall_reserve', np.where(strong_visual, 'strong_reference_tail', 'corroborated_tail_signal'))
    ranked.loc[rescue_to_borderline & ~exact_hash, 'dropoff_band'] = 'borderline'
    ranked.loc[rescue_to_borderline & ~exact_hash, 'dropoff_reason'] = rescue_reason[rescue_to_borderline & ~exact_hash]
    ranked['rescued_for_recall'] = rescue_to_borderline | exact_hash

    high_df = ranked[ranked['dropoff_band'] == 'high'].copy()
    edge_df = ranked[ranked['dropoff_band'] == 'borderline'].copy()
    low_df = ranked[ranked['dropoff_band'] == 'low'].copy()
    lane_counts = ranked.groupby(['retrieval_lane', 'dropoff_band']).size().to_dict()
    print(f'[Drop-off] Lane-aware boundaries: High {len(high_df):,}, Borderline {len(edge_df):,}, Deferred Low {len(low_df):,}; detail: {lane_counts}.')
    return high_df, edge_df, low_df

def _fast_contextual_score(row: dict, audit_context: dict) -> dict:
    '''Local contextual score: exact tag/term coverage, source consensus, rank and vector affinity. No model call.'''
    terms = set(_search_terms(audit_context))
    requested_tags = {tag.casefold() for tag in _exact_tags(audit_context)}
    tokens = set(re.findall(r'[a-z0-9]+', f"{row.get('gemini_description') or ''} {row.get('asset_filename') or ''}".casefold()))
    row_tags = {str(tag).casefold() for tag in (row.get('vision_tags') or [])}
    keyword_coverage = len(tokens.intersection(terms)) / max(1, len(terms))
    tag_coverage = len(row_tags.intersection(requested_tags)) / max(1, len(requested_tags))
    sources = _source_names(row)
    source_consensus = min(1.0, len(sources) / 3.0)
    rank_score = _as_float(row.get('retrieval_rank_percentile'), 0.5)
    reference_distance = _as_float(row.get('reference_vector_distance'), 9.0)
    text_distance = _as_float(row.get('text_vector_distance'), 9.0)
    best_distance = min(reference_distance, text_distance)
    vector_affinity = max(0.0, min(1.0, (0.65 - best_distance) / 0.65))
    negative_distance = _as_float(row.get('negative_vector_distance'), 1.0)
    negative_penalty = max(0.0, min(0.12, (0.35 - negative_distance) * 0.35))
    exact_bonus = 0.20 if row.get('exact_content_match') else 0.0
    score = (0.34 * rank_score + 0.22 * source_consensus + 0.20 * keyword_coverage + 0.14 * tag_coverage + 0.10 * vector_affinity + exact_bonus - negative_penalty)
    return {
        'fast_context_score': max(0.0, min(1.0, score)),
        'context_keyword_coverage': keyword_coverage,
        'context_tag_coverage': tag_coverage,
        'context_source_consensus': source_consensus,
        'context_vector_affinity': vector_affinity,
    }

async def run_semantic_reranking_and_filter(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_context: dict, max_workers: int = None) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''Fast deterministic contextual reranking; Gemini remains reserved for final image-level true/false inference.'''
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    candidates_df = candidates_df.drop_duplicates(subset=['content_hash', 'gcs_raw_path'], keep='first').copy()
    high, edge, low = [], [], []
    for row in candidates_df.to_dict('records'):
        scored = {**row, **_fast_contextual_score(row, audit_context)}
        sources = _source_names(scored)
        is_exact = bool(scored.get('exact_content_match'))
        has_strong_visual = 'reference_vector' in sources and _as_float(scored.get('reference_vector_distance'), 9.0) <= SEARCH_TUNING['visual_safeguard_distance']
        has_recall_reserve = bool(scored.get('recall_reserve_source'))
        band = str(scored.get('dropoff_band') or 'borderline')
        context_score = scored['fast_context_score']
        scored['cross_encoder_score'] = int(round(context_score * 100))
        scored['contextual_rerank_method'] = 'local_deterministic'
        scored['relevance_score'] = 0.65 * context_score + 0.35 * _as_float(scored.get('retrieval_rank_percentile'), 0.5)
        if is_exact:
            scored['relevance_score'] = 1.0
            high.append(scored)
        elif band == 'high' and context_score >= 0.45:
            high.append(scored)
        elif has_strong_visual or has_recall_reserve or (band in {'high', 'borderline'} and context_score >= 0.24):
            edge.append(scored)
        else:
            low.append(scored)
    def as_sorted_frame(rows):
        return pd.DataFrame(rows).sort_values(['relevance_score', 'fast_context_score'], ascending=False).reset_index(drop=True) if rows else pd.DataFrame()
    high_df, edge_df, low_df = as_sorted_frame(high), as_sorted_frame(edge), as_sorted_frame(low)
    print(f'[Contextual rerank] Local scoring completed for {len(candidates_df):,} hashes: High {len(high_df):,}, Borderline {len(edge_df):,}, Low {len(low_df):,}.')
    return high_df, edge_df, low_df

def enforce_zero_false_positives_rules(df: pd.DataFrame) -> pd.DataFrame:
    '''Preserve model-positive low-confidence matches as reviewable candidates instead of silently discarding recall.'''
    if df.empty:
        return df
    def label_confidence(row):
        confidence = int(_as_float(row.get('match_confidence'), 0))
        is_match = str(row.get('matches_criteria', False)).casefold() in {'true', '1', 'yes'}
        evaluation_error = str(row.get('evaluation_status', '')).casefold() == 'error' or bool(row.get('error'))
        row['requires_review'] = bool(evaluation_error or (is_match and confidence < 75))
        row['confidence_band'] = 'error' if evaluation_error else ('high' if confidence >= 95 else ('confirmed' if confidence >= 75 else ('review' if confidence >= SEARCH_TUNING['minimum_review_confidence'] else 'low')))
        return row
    return df.apply(label_confidence, axis=1)

def _hybrid_visual_audit_config(audit_config: dict) -> dict:
    '''Add retrieval-independent visual decision evidence for the hybrid evaluator without changing the comprehensive-scan executor.'''
    prepared = dict(audit_config)
    signatures = audit_config.get('primary_visual_signatures', [])
    variations = audit_config.get('allowed_variations', [])
    confusions = audit_config.get('disqualifying_confusions', [])
    evidence_protocol = f'''\n\nHYBRID VISUAL DECISION PROTOCOL (pixel evidence only):\n- Evaluation strategy: {audit_config.get('evaluation_strategy', 'semantic_visual_presence')}\n- Required primary signatures: {'; '.join(signatures) if signatures else 'Use the explicit inclusion criteria.'}\n- Allowed variations: {'; '.join(variations) if variations else 'None unless an inclusion criterion permits it.'}\n- Disqualifying confusions: {'; '.join(confusions) if confusions else 'Apply the explicit exclusion criteria.'}\n- Do not let retrieval score, filename, Gemini description, vision tags, or source-arm rank determine a True verdict. They selected the candidate; only visible image evidence decides it.\n- Return true only when every required inclusion/signature is visibly supported and no exclusion/confusion is visible. Return false for a near miss, a different subject/identity, insufficient visual evidence, or a conflict with any exclusion.\n'''
    prepared['audit_instructions'] = (audit_config.get('audit_instructions', '') + evidence_protocol).strip()
    return prepared

async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_config: dict, max_workers: int = None, reference_image_path: Optional[str] = None) -> pd.DataFrame:
    '''Audit bounded batches of unique hashes, reusing the existing per-image Gemini evaluator and caching successful evaluations.'''
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()
    strategy = str(audit_config.get('evaluation_strategy', '')).strip().casefold()
    if strategy == 'content_hash_exact' and bool(audit_config.get('exact_hash_shortcut_authorized', False)) and reference_image_path:
        exact_candidates = candidates_df[candidates_df.get('exact_content_match', pd.Series(False, index=candidates_df.index)).fillna(False)].drop_duplicates(subset=['content_hash', 'gcs_raw_path'], keep='first').copy()
        if exact_candidates.empty:
            print('[Exact identity] No matching content hash was found; no visual-model call is needed.')
            return pd.DataFrame()
        exact_candidates['matches_criteria'] = True
        exact_candidates['match_confidence'] = 100
        exact_candidates['match_rationale'] = 'True: the candidate content_hash is identical to the uploaded reference bytes.'
        exact_candidates['visual_analysis_step_by_step'] = 'Deterministic content-hash identity match; visual inference was skipped.'
        exact_candidates['evaluation_method'] = 'content_hash_exact'
        exact_candidates['evaluation_status'] = 'completed'
        print(f'[Exact identity] Verified {len(exact_candidates):,} unique content hash(es) without a visual-model call.')
        return enforce_zero_false_positives_rules(exact_candidates.sort_values('relevance_score', ascending=False).reset_index(drop=True))
    candidates_df, deferred_df = _select_diverse_candidates(candidates_df, SEARCH_TUNING['max_visual_audit_candidates'])
    if not deferred_df.empty:
        print(f'[Visual-audit budget] Auditing {len(candidates_df):,} unique hashes; {len(deferred_df):,} more are retained in the ranked retrieval dataframe for a later recall sweep.')
        globals()['deferred_visual_candidates_global'] = deferred_df.copy()
    else:
        globals()['deferred_visual_candidates_global'] = pd.DataFrame(columns=candidates_df.columns)
    candidates = candidates_df.to_dict('records')
    reference_key = '|'.join(_reference_hashes(reference_image_path)) if reference_image_path else 'text-only'
    visual_audit_config = _hybrid_visual_audit_config(audit_config)
    config_key = hashlib.sha256((json.dumps(visual_audit_config, sort_keys=True, default=str) + reference_key).encode()).hexdigest()
    worker_count = max_workers or SEARCH_TUNING['inference_workers']
    loop = asyncio.get_running_loop()
    def get_reference_part():
        return types.Part.from_bytes(data=process_transparency(_read_image_bytes(reference_image_path)), mime_type='image/png')
    async def audit_one(asset, executor, reference_part):
        key = (_content_key(asset), config_key)
        if key in _AUDIT_RESULT_CACHE:
            return {**asset, **_AUDIT_RESULT_CACHE[key], 'audit_cache_hit': True}
        result = await run_llm_audit_single(asset, visual_audit_config, _executor=executor, reference_image_part=reference_part)
        extracted = {key: value for key, value in result.items() if key not in asset}
        if not extracted.get('error'):
            _AUDIT_RESULT_CACHE[key] = extracted
        return result
    started = time.time()
    results = []
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        reference_part = await loop.run_in_executor(executor, get_reference_part) if reference_image_path else None
        for start in range(0, len(candidates), SEARCH_TUNING['inference_batch_size']):
            batch = candidates[start:start + SEARCH_TUNING['inference_batch_size']]
            results.extend(await asyncio.gather(*[audit_one(asset, executor, reference_part) for asset in batch]))
            print(f'[Visual audit] {min(start + len(batch), len(candidates))}/{len(candidates)} unique hashes completed')
    results_df = enforce_zero_false_positives_rules(pd.DataFrame(results))
    results_df = results_df.sort_values('relevance_score', ascending=False).reset_index(drop=True)
    print(f'[Visual audit] Completed {len(results_df):,} unique hashes in {time.time() - started:.2f}s.')
    return results_df

def build_audited_hash_verdict_map(results_df: pd.DataFrame) -> pd.DataFrame:
    '''Keep one LLM verdict per content hash; this compact map is joined onto every duplicate physical row only when pages are requested.'''
    columns = ['content_hash', 'matches_criteria', 'match_confidence', 'match_rationale', 'visual_analysis_step_by_step', 'evaluation_status', 'requires_review', 'confidence_band']
    if results_df is None or results_df.empty or 'content_hash' not in results_df.columns:
        return pd.DataFrame(columns=columns)
    available = [column for column in columns if column in results_df.columns]
    verdicts = results_df[available].copy()
    verdicts['content_hash'] = verdicts['content_hash'].fillna('').astype(str)
    verdicts = verdicts[verdicts['content_hash'].ne('')].copy()
    if verdicts.empty:
        return pd.DataFrame(columns=columns)
    completed_first = verdicts.get('evaluation_status', pd.Series('', index=verdicts.index)).astype(str).str.casefold().eq('completed').astype(int)
    verdicts['_completed_first'] = completed_first
    if 'match_confidence' not in verdicts.columns:
        verdicts['match_confidence'] = 0
    verdicts = verdicts.sort_values(['_completed_first', 'match_confidence'], ascending=[False, False]).drop_duplicates('content_hash', keep='first').drop(columns=['_completed_first']).reset_index(drop=True)
    for column in columns:
        if column not in verdicts.columns:
            verdicts[column] = None
    verdicts['matches_criteria'] = verdicts['matches_criteria'].astype(str).str.casefold().isin({'true', '1', 'yes'})
    globals()['audited_hash_verdicts_global'] = verdicts[columns].copy()
    globals()['audited_content_hashes_global'] = verdicts['content_hash'].tolist()
    return verdicts[columns]

async def summarize_audited_instances(results_df: pd.DataFrame) -> pd.DataFrame:
    '''Count physical assets for every completed/error Boolean verdict without expanding duplicate rows in notebook memory.'''
    verdicts = build_audited_hash_verdict_map(results_df)
    if verdicts.empty:
        return pd.DataFrame(columns=['content_hash', 'matches_criteria', 'evaluation_status', 'instance_count'])
    hashes = verdicts['content_hash'].tolist()
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        rows = await db.fetch(
            f'''SELECT content_hash, COUNT(*) AS instance_count
                FROM {DB_SCHEMA}.visual_assets
                WHERE content_hash = ANY(CAST($1 AS text[]))
                GROUP BY content_hash
                ORDER BY instance_count DESC, content_hash''',
            hashes,
        )
    summary = pd.DataFrame([dict(row) for row in rows])
    if not summary.empty:
        summary['content_hash'] = summary['content_hash'].astype(str)
        summary = summary.merge(verdicts[['content_hash', 'matches_criteria', 'evaluation_status', 'requires_review']], on='content_hash', how='left', validate='one_to_one')
    globals()['audited_instance_summary_global'] = summary
    print(f'[Verdict propagation] {len(verdicts):,} unique-hash LLM verdicts map to {int(summary["instance_count"].sum()) if not summary.empty else 0:,} physical asset/page rows. Use fetch_audited_instance_page() to retrieve the propagated rows safely.')
    return summary

async def fetch_audited_instance_page(page_size: Optional[int] = None, cursor: Optional[Tuple[str, str]] = None, verdicts_df: Optional[pd.DataFrame] = None) -> Tuple[pd.DataFrame, Optional[Tuple[str, str]]]:
    '''Return a keyset-paginated page of physical assets with the stored unique-hash LLM Boolean verdict copied onto each page URL.'''
    verdicts = verdicts_df.copy() if verdicts_df is not None else globals().get('audited_hash_verdicts_global', pd.DataFrame())
    if verdicts is None or verdicts.empty or 'content_hash' not in verdicts.columns:
        return pd.DataFrame(), None
    verdicts = verdicts.drop_duplicates('content_hash', keep='first').copy()
    verdicts['content_hash'] = verdicts['content_hash'].astype(str)
    hashes = verdicts['content_hash'].tolist()
    page_size = min(int(page_size or SEARCH_TUNING['instance_page_size']), SEARCH_TUNING['instance_page_size'])
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        if cursor:
            rows = await db.fetch(
                f'''SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE content_hash = ANY(CAST($1 AS text[])) AND (content_hash, asset_id::text) > ($2, $3)
                    ORDER BY content_hash, asset_id::text LIMIT $4''',
                hashes, cursor[0], cursor[1], page_size + 1,
            )
        else:
            rows = await db.fetch(
                f'''SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE content_hash = ANY(CAST($1 AS text[]))
                    ORDER BY content_hash, asset_id::text LIMIT $2''',
                hashes, page_size + 1,
            )
    has_more = len(rows) > page_size
    page = pd.DataFrame([dict(row) for row in rows[:page_size]])
    if page.empty:
        return page, None
    page['content_hash'] = page['content_hash'].astype(str)
    page = page.merge(verdicts, on='content_hash', how='left', validate='many_to_one')
    next_cursor = (str(page.iloc[-1]['content_hash']), str(page.iloc[-1]['asset_id'])) if has_more else None
    return page, next_cursor

async def summarize_verified_instances(results_df: pd.DataFrame) -> pd.DataFrame:
    '''Count all physical asset rows for verified hashes without materialising duplicate rows in the notebook.'''
    if results_df is None or results_df.empty or 'content_hash' not in results_df.columns:
        return pd.DataFrame(columns=['content_hash', 'instance_count'])
    verified = results_df[results_df['matches_criteria'].astype(str).str.casefold().isin({'true', '1', 'yes'})].copy()
    hashes = [str(value) for value in verified['content_hash'].dropna().unique() if str(value)]
    if not hashes:
        return pd.DataFrame(columns=['content_hash', 'instance_count'])
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        rows = await db.fetch(
            f'''SELECT content_hash, COUNT(*) AS instance_count
                FROM {DB_SCHEMA}.visual_assets
                WHERE content_hash = ANY($1::text[])
                GROUP BY content_hash
                ORDER BY instance_count DESC, content_hash''',
            hashes,
        )
    summary = pd.DataFrame([dict(row) for row in rows])
    globals()['verified_content_hashes_global'] = hashes
    globals()['verified_instance_summary_global'] = summary
    print(f'[Instance expansion] {len(hashes):,} verified content hashes represent {int(summary['instance_count'].sum()) if not summary.empty else 0:,} physical asset rows. Use fetch_verified_instance_page() to page them safely.')
    return summary

async def fetch_verified_instance_page(content_hashes: Optional[List[str]] = None, page_size: Optional[int] = None, cursor: Optional[Tuple[str, str]] = None) -> Tuple[pd.DataFrame, Optional[Tuple[str, str]]]:
    '''Return one deterministic page of all instances for verified hashes; cursor is the prior returned (content_hash, asset_id).'''
    hashes = content_hashes or globals().get('verified_content_hashes_global', [])
    if not hashes:
        return pd.DataFrame(), None
    page_size = min(int(page_size or SEARCH_TUNING['instance_page_size']), SEARCH_TUNING['instance_page_size'])
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        if cursor:
            rows = await db.fetch(
                f'''SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE content_hash = ANY($1::text[]) AND (content_hash, asset_id::text) > ($2, $3)
                    ORDER BY content_hash, asset_id::text LIMIT $4''',
                hashes, cursor[0], cursor[1], page_size,
            )
        else:
            rows = await db.fetch(
                f'''SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE content_hash = ANY($1::text[])
                    ORDER BY content_hash, asset_id::text LIMIT $2''',
                hashes, page_size,
            )
    page = pd.DataFrame([dict(row) for row in rows])
    next_cursor = (str(page.iloc[-1]['content_hash']), str(page.iloc[-1]['asset_id'])) if len(page) == page_size else None
    return page, next_cursor


In [ ]:
# TEST RUN (Stage 1): Generate Audit Rules & Configuration
# Run this cell to upload your reference image and generate the rule configuration.
import nest_asyncio
nest_asyncio.apply()

# Dynamic Reference Image Detector (Triggered via Colab Interactive Upload)
reference_image_path = None
try:
    from google.colab import files
    print("[OPTIONAL] Upload a reference image for comparative compliance audit:")
    uploaded = files.upload()
    if uploaded:
        reference_image_path = list(uploaded.keys())[0]
        print(f" Reference image uploaded: {reference_image_path}")
    else:
        print(" No reference image uploaded. Running text-only audit goal.")
except Exception as e:
    reference_image_path = None

# Enterprise Audit Goal (Modify this as needed)
TEST_AUDIT_GOAL = "Find all pages with this exact image"

# Step 1: Generate configuration rules
audit_config_global = None
try:
    audit_config_global = await generate_audit_config_only(
        user_goal=TEST_AUDIT_GOAL,
        reference_image_path=reference_image_path
    )
    print("💡 TIP: You can inspect and tweak 'audit_config_global' directly in the cell below before running Stage 2.")
except Exception as e:
    print(f"Error generating audit configuration: {e}")


In [ ]:
# TEST RUN (Stage 2): Execute Visual Search & Parallel Audit
# Run this cell to execute retrieval, segmentation, and LLM inference.
# You can uncomment and modify rules below to calibrate config before executing.

if 'audit_config_global' in globals() and audit_config_global is not None:
    # OPTIONAL CALIBRATION TUNING:
    # If the generated AI rules were slightly off, you can uncomment and edit them here:
    # audit_config_global["inclusion_criteria"] = [
    #     "The image must contain the legacy Google Pay logo featuring the interlocking loops design."
    # ]
    # audit_config_global["exclusion_criteria"] = [
    #     "Exclude images containing the current compliant Google Pay GPay wordmark button logo."
    # ]

    df_results_global = None
    try:
        df_results_global = await run_full_test_bench_pipeline_execution(
            audit_config=audit_config_global,
            reference_image_path=reference_image_path
        )
    except Exception as e:
        print(f"Error executing test bench pipeline: {e}")
else:
    print(" Please run Stage 1 cell first to generate 'audit_config_global'.")
